# Chuẩn bị packages native cho notebook 1 và notebook 3

Chạy trên Kaggle với GPU L4, Internet Off. Chỉ gắn Dataset **packages**, không cần models/PDF.
`PACKAGES_DIR` là bắt buộc về dữ liệu: đường dẫn đã điền theo Dataset hiện dùng của bạn;
nếu mount khác thì sửa lại. Đặt `''` chỉ khi muốn tự tìm qua manifest, không phải bỏ packages.
Hai phần bật/tắt độc lập: FlashAttention cho UniHIR và native wheels TongGu/MMCV cho notebook 3.
Lần đầu vẫn cần build; các phiên sau dùng wheel đã kiểm tra. Không load model hay chạy OCR.

In [ ]:
from pathlib import Path
if not Path('/kaggle/input').is_dir() or not Path('/kaggle/working').is_dir():
    raise RuntimeError('Notebook này chỉ chạy trên Kaggle với GPU L4 và Internet Off.')
PACKAGES_DIR = '/kaggle/input/datasets/mduy2911/nlp-package' # sửa nếu mount khác; '' để tự tìm
PREPARE_FLASHATTENTION = True
PREPARE_NOTEBOOK3 = True
RUNTIME_DIR = '/tmp/vtm_image_runtime'
OCR_RUNTIME_DIR = '/tmp/vtm_kaggle_runtime'
OUTPUT_DIR = '/kaggle/working/flash_attention_build'
OCR_OUTPUT_DIR = '/kaggle/working/ocr_native_build'
CACHE_INPUT = ''   # tùy chọn cho FlashAttention; cả hai loại cache trong packages được tự tìm
ALLOW_BUILD = True # False: chỉ dùng wheel có sẵn; dừng sớm nếu chưa có
MAX_BUILD_JOBS = 8 # bootstrap tự giảm theo CPU/RAM thực tế
NVCC_THREADS = 2
OCR_BUILD_TIMEOUT_SECONDS = 21600

## Bootstrap offline được nhúng

Chỉ dùng source đã có trong packages, kiểm tra hash và ABI. Không tải wheel từ Internet.

In [ ]:
import hashlib, importlib.util, json, os
FLASH_RUNTIME_SOURCE = '"""Build isolated restoration overlays from staged files, entirely offline.\n\nRun on Kaggle Linux with the working Python 3.13 / Torch CUDA 13 kernel.\nFlashAttention is compiled for the detected GPU against that same Torch ABI.\nNo weights are loaded. Notebook 1 separately runs its worker/GPU probes.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.metadata as metadata\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tarfile\nimport time\nimport zipfile\n\n\nFLASH_CUDA_PROBE = (\n    \'import torch; from flash_attn import flash_attn_varlen_func, flash_attn_func; \'\n    \'q=torch.randn(16,2,64,device="cuda",dtype=torch.bfloat16); \'\n    \'cu=torch.tensor([0,16],device="cuda",dtype=torch.int32); \'\n    \'out=flash_attn_varlen_func(q,q,q,cu,cu,16,16); \'\n    \'fixed=flash_attn_func(q[None],q[None],q[None]); \'\n    \'torch.cuda.synchronize(); assert out.shape==q.shape and torch.isfinite(out).all(); \'\n    \'assert fixed.shape==q[None].shape and torch.isfinite(fixed).all()\'\n)\n\n\ndef flash_cache_identity(torch, source):\n    return dict(version=1, source_sha256=sha256(source),\n        python=list(sys.version_info[:2]), platform=sys.platform, machine=platform.machine(),\n        libc=list(platform.libc_ver()), torch=torch.__version__, cuda=torch.version.cuda,\n        cxx11_abi=bool(torch._C._GLIBCXX_USE_CXX11_ABI),\n        capability=list(torch.cuda.get_device_capability()))\n\n\ndef flash_cache_output():\n    return Path(os.environ.get(\'VTM_FLASH_CACHE_OUTPUT\', \'/kaggle/working/flash_attention_cache\'))\n\n\ndef flash_cache_roots(packages_dir=None):\n    roots=[]\n    if os.environ.get(\'VTM_FLASH_CACHE_INPUT\'):\n        roots.append(Path(os.environ[\'VTM_FLASH_CACHE_INPUT\']))\n    if packages_dir is not None:\n        roots.append(Path(packages_dir)/\'image_restoration/flash_attention_cache\')\n    roots.append(flash_cache_output())\n    return roots\n\n\ndef find_flash_cache(identity, roots):\n    """Require both runtime identity and wheel integrity; never guess by filename."""\n    from packaging.tags import sys_tags\n    from packaging.utils import parse_wheel_filename\n    key=hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()\n    for root in roots:\n        for record in (Path(root)/key/\'flash_wheel.json\', Path(root)/\'flash_wheel.json\'):\n            if not record.is_file():continue\n            data=json.loads(record.read_text(encoding=\'utf8\'))\n            if data.get(\'identity\')!=identity:continue\n            if not data.get(\'cuda_probe_verified\'):continue\n            wheel=contained(record.parent,data[\'wheel\'])\n            if (wheel.suffix!=\'.whl\' or not wheel.is_file() or\n                    wheel.stat().st_size!=data[\'bytes\'] or sha256(wheel)!=data[\'sha256\']):\n                raise ValueError(f\'Corrupt FlashAttention cache: {record}\')\n            if not set(sys_tags()).intersection(parse_wheel_filename(wheel.name)[3]):\n                raise ValueError(f\'Incompatible FlashAttention wheel tags: {wheel.name}\')\n            return wheel,data\n    return None\n\n\ndef save_flash_cache(identity, wheel, root):\n    """Call only after the extension has passed an actual CUDA probe."""\n    key=hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()\n    destination=Path(root)/key\n    destination.mkdir(parents=True,exist_ok=True)\n    target=destination/wheel.name\n    digest=sha256(wheel)\n    if wheel.resolve()!=target.resolve():\n        temporary=target.with_suffix(\'.whl.tmp\')\n        shutil.copyfile(wheel,temporary)\n        if sha256(temporary)!=digest:raise ValueError(\'FlashAttention cache copy checksum mismatch\')\n        temporary.replace(target)\n    data=dict(identity=identity,wheel=wheel.name,sha256=digest,bytes=target.stat().st_size,\n        cuda_probe_verified=True,model_inference_verified=False)\n    record=destination/\'flash_wheel.json\';temporary=record.with_suffix(\'.tmp\')\n    temporary.write_text(json.dumps(data,indent=2)+\'\\n\',encoding=\'utf8\');temporary.replace(record)\n    print(f\'FlashAttention cache saved: {destination}; retain this folder for the next session\',flush=True)\n    return data\n\n\ndef export_flash_cache(packages_dir, runtime_dir, output_dir=None):\n    """Rescue a wheel from an existing session, including older runtime builders."""\n    import torch\n    root=Path(packages_dir)/\'image_restoration\'\n    plan=json.loads((root/\'manifests/unihir_runtime.json\').read_text(encoding=\'utf8\'))\n    validate_plan(root,plan)\n    for name,expected in plan[\'kernel_reuse\'].items():\n        if metadata.version(name)!=expected:raise RuntimeError(f\'Kernel changed: {name}\')\n    source=contained(root,plan[\'native_sources\'][0][\'path\'])\n    identity=flash_cache_identity(torch,source)\n    for folder in sorted((Path(runtime_dir)/\'environments\').glob(\'unihir_restoration_*\')):\n        receipt=folder/\'install.json\'\n        if not receipt.is_file():continue\n        installed=json.loads(receipt.read_text(encoding=\'utf8\'))\n        for native in installed.get(\'native\',[]):\n            if native.get(\'sm\')!=\'\'.join(map(str,identity[\'capability\'])):continue\n            wheel=contained(folder/\'native_build/wheels\',native[\'wheel\'])\n            if not wheel.is_file() or sha256(wheel)!=native[\'sha256\']:continue\n            subprocess.run([str(folder/\'bin/python\'),\'-c\',FLASH_CUDA_PROBE],\n                env=offline_environment(),check=True,timeout=120)\n            save_flash_cache(identity,wheel,output_dir or flash_cache_output())\n            return\n    raise FileNotFoundError(\'No completed FlashAttention runtime/wheel found; wait for the current build to finish\')\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(8 * 1024**2), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef contained(root, relative):\n    root = Path(root).resolve()\n    path = (root / relative).resolve()\n    if not path.is_relative_to(root):\n        raise ValueError(f\'Path outside bundle: {relative}\')\n    return path\n\n\ndef native_build_resources(*, cpus=None, available_bytes=None, environ=None):\n    """Bound compiler parallelism using CPU affinity and effective available RAM."""\n    environ = os.environ if environ is None else environ\n    if cpus is None:\n        cpus = len(os.sched_getaffinity(0)) if hasattr(os,\'sched_getaffinity\') else os.cpu_count() or 1\n    if available_bytes is None:\n        candidates=[]\n        try:\n            values=dict(line.split(\':\',1) for line in Path(\'/proc/meminfo\').read_text().splitlines())\n            candidates.append(int(values[\'MemAvailable\'].split()[0])*1024)\n        except (OSError,KeyError,ValueError):pass\n        try:\n            limit=Path(\'/sys/fs/cgroup/memory.max\').read_text().strip()\n            used=int(Path(\'/sys/fs/cgroup/memory.current\').read_text())\n            if limit.isdigit():candidates.append(max(0,int(limit)-used))\n        except (OSError,ValueError):pass\n        available_bytes=min(candidates) if candidates else None\n    requested=int(environ.get(\'VTM_FLASH_MAX_JOBS\',\'8\'))\n    threads=int(environ.get(\'VTM_FLASH_NVCC_THREADS\',\'2\'))\n    if not 1<=requested<=16 or not 1<=threads<=4:\n        raise ValueError(\'VTM_FLASH_MAX_JOBS must be 1..16; VTM_FLASH_NVCC_THREADS must be 1..4\')\n    # Reserve 25% RAM and budget 12 GiB per compile job. Unknown RAM stays at <=2.\n    memory_jobs=max(1,int(available_bytes*.75)//(12*1024**3)) if available_bytes is not None else 2\n    jobs=max(1,min(requested,max(1,cpus//threads),memory_jobs))\n    return dict(cpu_affinity=cpus,available_ram_bytes=available_bytes,max_jobs=jobs,nvcc_threads=threads,\n                requested_max_jobs=requested,memory_budget_per_job_gib=12,available_ram_fraction=.75)\n\n\ndef offline_environment():\n    env = dict(os.environ)\n    for key in (\'PYTHONPATH\', \'PYTHONHOME\', \'PYTHONUSERBASE\'):\n        env.pop(key, None)\n    env.update(PYTHONNOUSERSITE=\'1\', PYTHONUTF8=\'1\', PIP_NO_INDEX=\'1\',\n               PIP_DISABLE_PIP_VERSION_CHECK=\'1\', HF_HUB_OFFLINE=\'1\',\n               TRANSFORMERS_OFFLINE=\'1\', HF_DATASETS_OFFLINE=\'1\',\n               HF_HUB_DISABLE_TELEMETRY=\'1\', MPLBACKEND=\'Agg\',\n               FLASH_ATTENTION_FORCE_BUILD=\'TRUE\')\n    resources=native_build_resources(environ=env)\n    env.update(MAX_JOBS=str(resources[\'max_jobs\']),NVCC_THREADS=str(resources[\'nvcc_threads\']))\n    return env\n\n\ndef validate_plan(root, plan):\n    from packaging.tags import sys_tags\n    from packaging.utils import parse_wheel_filename\n    tags = set(sys_tags())\n    for record in plan[\'wheels\'] + plan.get(\'native_sources\', []):\n        path = contained(root, record[\'path\'])\n        if not path.is_file() or path.stat().st_size != record[\'bytes\'] or sha256(path) != record[\'sha256\']:\n            raise ValueError(f\'Missing or corrupt offline payload: {path}\')\n        if path.suffix == \'.whl\' and not tags.intersection(parse_wheel_filename(path.name)[3]):\n            raise RuntimeError(f\'Incompatible wheel: {path.name}\')\n    if plan.get(\'dependency_problems\') or plan.get(\'state\') != \'prepared_unverified\':\n        raise RuntimeError(f\'Bundle dependency plan is incomplete: {plan.get("dependency_problems")}\')\n\n\ndef build_native(python, source, folder, env, packages_dir=None):\n    """Only a verified PyPI sdist, including CUTLASS; never a git/network build."""\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'FlashAttention build/probe needs an available CUDA GPU\')\n    major, minor = torch.cuda.get_device_capability()\n    identity=flash_cache_identity(torch,source)\n    cached=find_flash_cache(identity,flash_cache_roots(packages_dir))\n    if cached:\n        wheel,record=cached\n        print(f\'Reusing FlashAttention wheel; compilation skipped: {wheel}\',flush=True)\n        # Retain the existing runtime receipt layout used by notebook 0.\n        local_wheels=folder/\'native_build/wheels\';local_wheels.mkdir(parents=True,exist_ok=True)\n        local_wheel=local_wheels/wheel.name\n        if local_wheel.resolve()!=wheel.resolve():\n            shutil.copyfile(wheel,local_wheel)\n        if sha256(local_wheel)!=record[\'sha256\']:raise ValueError(\'Cached wheel copy changed\')\n        wheel=local_wheel\n        subprocess.run([str(python),\'-m\',\'pip\',\'--isolated\',\'install\',\'--no-index\',\n                        \'--no-deps\',\'--ignore-installed\',str(wheel)],env=env,check=True)\n        return dict(wheel=wheel.name,sha256=record[\'sha256\'],sm=f\'{major}{minor}\',\n                    cached=True,wheel_path=str(wheel),build_seconds=0)\n    if os.environ.get(\'VTM_FLASH_REQUIRE_CACHE\')==\'1\':\n        raise FileNotFoundError(\'No compatible verified FlashAttention wheel cache. \'\n            \'Attach flash_attention_cache and set VTM_FLASH_CACHE_INPUT; compilation is disabled.\')\n    env = dict(env, FLASH_ATTN_CUDA_ARCHS=f\'{major}{minor}\',\n               TORCH_CUDA_ARCH_LIST=f\'{major}.{minor}\')\n    scratch = folder/\'native_build\'\n    unpacked = scratch/\'source\'\n    unpacked.mkdir(parents=True, exist_ok=True)\n    with tarfile.open(source) as archive:\n        archive.extractall(unpacked, filter=\'data\')\n    roots = [p for p in unpacked.iterdir() if p.is_dir()]\n    if len(roots) != 1 or not (roots[0]/\'csrc/cutlass/include/cutlass/cutlass.h\').is_file():\n        raise ValueError(\'Incomplete FlashAttention source/CUTLASS bundle\')\n    if (roots[0]/\'.git\').exists():\n        raise ValueError(\'A git checkout could run network submodule updates; refusing\')\n    wheels = scratch/\'wheels\'\n    wheels.mkdir(exist_ok=True)\n    resources=native_build_resources(environ=env)\n    env.update(MAX_JOBS=str(resources[\'max_jobs\']),NVCC_THREADS=str(resources[\'nvcc_threads\']))\n    config=dict(resources,source_sha256=sha256(source),torch=torch.__version__,cuda=torch.version.cuda,\n                gpu_capability=[major,minor],torch_cuda_arch_list=env[\'TORCH_CUDA_ARCH_LIST\'],\n                flash_attn_cuda_archs=env[\'FLASH_ATTN_CUDA_ARCHS\'],model_weights_loaded=False)\n    (scratch/\'build_config.json\').write_text(json.dumps(config,indent=2)+\'\\n\',encoding=\'utf8\')\n    started=time.monotonic()\n    # Build against inherited Torch, using the existing Linux toolchain. Logs stay in /tmp.\n    with (scratch/\'build.log\').open(\'w\', encoding=\'utf8\') as log:\n        command = [str(python), \'-m\', \'pip\', \'--isolated\', \'wheel\', \'--verbose\', \'--no-index\',\n                   \'--no-deps\', \'--no-build-isolation\', \'--wheel-dir\', str(wheels), str(roots[0])]\n        log.write(json.dumps(dict(config,command=command))+\'\\n\');log.flush()\n        print(f\'Building FlashAttention for sm_{major}{minor}; MAX_JOBS={resources["max_jobs"]}, \'\n              f\'NVCC_THREADS={resources["nvcc_threads"]}; log: {scratch / "build.log"}\', flush=True)\n        process=subprocess.Popen(command,env=env,stdout=log,stderr=subprocess.STDOUT)\n        last_notice=started\n        def progress(state):\n            with (scratch/\'build.log\').open(\'rb\') as stream:\n                stream.seek(0,2);stream.seek(max(0,stream.tell()-32768));tail=stream.read().decode(\'utf8\',errors=\'replace\')\n            counts=re.findall(r\'\\[(\\d+)/(\\d+)\\]\',tail)\n            record=dict(status=state,pid=process.pid,seconds=round(time.monotonic()-started,2),\n                        max_jobs=resources[\'max_jobs\'],nvcc_threads=resources[\'nvcc_threads\'],\n                        ninja_finished=int(counts[-1][0]) if counts else None,\n                        ninja_total=int(counts[-1][1]) if counts else None)\n            target=scratch/\'build_progress.json\';temporary=target.with_suffix(\'.tmp\')\n            temporary.write_text(json.dumps(record,indent=2)+\'\\n\',encoding=\'utf8\');temporary.replace(target)\n            return record\n        try:\n            progress(\'running\')\n            while process.poll() is None:\n                if time.monotonic()-last_notice>=30:\n                    record=progress(\'running\')\n                    count=f\'{record["ninja_finished"]}/{record["ninja_total"]}\' if record[\'ninja_total\'] else \'compiler initialization\'\n                    print(f\'FlashAttention build: {count}, {record["seconds"]:.0f}s, \'\n                          f\'{resources["max_jobs"]} jobs; see build.log\',flush=True)\n                    last_notice=time.monotonic()\n                time.sleep(.5)\n        except BaseException:\n            if process.poll() is None:process.kill();process.wait()\n            progress(\'interrupted\');raise\n        progress(\'compiled\' if process.returncode==0 else \'failed\')\n    if process.returncode:\n        raise RuntimeError(f\'Offline FlashAttention compilation failed. See {scratch / "build.log"}\')\n    built = list(wheels.glob(\'flash_attn-*.whl\'))\n    if len(built) != 1:\n        raise RuntimeError(f\'Expected one native wheel: {built}\')\n    subprocess.run([str(python), \'-m\', \'pip\', \'--isolated\', \'install\', \'--no-index\',\n                    \'--no-deps\', \'--ignore-installed\', str(built[0])], env=env, check=True)\n    return {\'wheel\': built[0].name, \'sha256\': sha256(built[0]), \'sm\': f\'{major}{minor}\',\n            \'cached\':False,\'wheel_path\':str(built[0]),\n            \'max_jobs\':resources[\'max_jobs\'],\'nvcc_threads\':resources[\'nvcc_threads\'],\n            \'build_seconds\':round(time.monotonic()-started,2)}\n\n\ndef prepare_runtime(packages_dir, runtime_dir, model):\n    if sys.platform != \'linux\' or platform.machine() != \'x86_64\' or sys.version_info[:2] != (3, 13):\n        raise RuntimeError(\'This bundle targets the reported Kaggle Linux x86_64 / Python 3.13 kernel\')\n    learned = model.startswith((\'docentr_\', \'degan_\'))\n    document = model.startswith((\'docres_\', \'docdiff_\', \'restormer_\')) or learned\n    if model not in (\'difftsr\', \'unihir_restoration\') and not document:\n        raise ValueError(model)\n    root = Path(packages_dir).resolve()/\'image_restoration\'\n    filename = (\'learned_documents_runtime.json\' if learned else \'document_runtime.json\'\n                if document else \'unihir_runtime.json\' if model == \'unihir_restoration\' else \'difftsr_runtime.json\')\n    plan = json.loads((root/\'manifests\'/filename).read_text(encoding=\'utf8\'))\n    validate_plan(root, plan)\n    for name, expected in plan[\'kernel_reuse\'].items():\n        actual = metadata.version(name)\n        if actual != expected:\n            raise RuntimeError(f\'Kernel changed: {name}={actual}; prepared against {expected}. \'\n                               \'Refresh the report/bundle instead of replacing kernel packages.\')\n    import torch\n    if torch.__version__ != plan[\'kernel_reuse\'][\'torch\']:\n        raise RuntimeError(\'Imported Torch differs from its metadata\')\n    # Recover a completed wheel from the previous bootstrap in this same session.\n    # This avoids a rebuild merely because the embedded bootstrap was updated.\n    if model==\'unihir_restoration\' and plan.get(\'native_sources\'):\n        identity=flash_cache_identity(torch,contained(root,plan[\'native_sources\'][0][\'path\']))\n        if not find_flash_cache(identity,flash_cache_roots(packages_dir)):\n            try:export_flash_cache(packages_dir,runtime_dir)\n            except FileNotFoundError:pass\n    key = hashlib.sha256(json.dumps({\'plan\': plan, \'python\': sys.version,\n        \'executable\': sys.executable, \'torch\': torch.__version__, \'cuda\': torch.version.cuda,\n        \'cxx11_abi\': torch._C._GLIBCXX_USE_CXX11_ABI,\n        \'capability\': torch.cuda.get_device_capability() if torch.cuda.is_available() else None,\n        \'bootstrap\': sha256(__file__)}, sort_keys=True).encode()).hexdigest()\n    folder = Path(runtime_dir).resolve()/\'environments\'/(model+\'_\'+key[:16])\n    if folder.is_relative_to(Path(\'/kaggle/input\')):\n        raise ValueError(\'Runtime must be writable, outside /kaggle/input\')\n    folder.mkdir(parents=True, exist_ok=True)\n    python = folder/\'bin/python\'\n    done = folder/\'install.json\'\n    env = offline_environment()\n    if learned:\n        env[\'KERAS_BACKEND\'] = \'torch\'\n    if not done.is_file():\n        # pip is inherited from the measured kernel; Ubuntu need not ship ensurepip.\n        subprocess.run([sys.executable, \'-m\', \'venv\', \'--without-pip\', \'--system-site-packages\', str(folder)],\n                       env=env, check=True)\n        wheels = [str(contained(root, record[\'path\'])) for record in plan[\'wheels\']]\n        if wheels:\n            subprocess.run([str(python), \'-m\', \'pip\', \'--isolated\', \'install\', \'--no-index\',\n                            \'--no-deps\', \'--ignore-installed\', *wheels], env=env, check=True)\n        native = [build_native(python, contained(root, record[\'path\']), folder, env,packages_dir=packages_dir)\n                  for record in plan.get(\'native_sources\', [])]\n        # Validate the actual native extension before publishing a reusable runtime.\n        code = (FLASH_CUDA_PROBE\n                if native else\n                \'import torch,numpy,cv2,einops,yaml; from skimage.filters import threshold_sauvola; \'\n                \'assert torch.from_numpy(numpy.ones((2,2),dtype=numpy.float32)).shape==(2,2)\'\n                if document and not learned else\n                \'import torch,numpy,cv2,vit_pytorch,h5py,keras; assert keras.backend.backend()=="torch"\'\n                if learned else\n                \'import zhconv,linear_attention_transformer,axial_positional_embedding; \'\n                \'import pytorch_lightning,omegaconf; assert zhconv.convert("abc","zh-cn")=="abc"\')\n        subprocess.run([str(python), \'-c\', code], env=env, check=True)\n        # Persist the expensive build before model loading/inference can fail.\n        for source_record,native_record in zip(plan.get(\'native_sources\',[]),native):\n            save_flash_cache(flash_cache_identity(torch,contained(root,source_record[\'path\'])),\n                Path(native_record[\'wheel_path\']),flash_cache_output())\n        data = {\'key\': key, \'native\': native, \'kernel_modified\': False,\n                \'weights_loaded\': False, \'model_inference_verified\': False}\n        temporary = done.with_suffix(\'.tmp\')\n        temporary.write_text(json.dumps(data, indent=2)+\'\\n\', encoding=\'utf8\')\n        temporary.replace(done)\n    return str(python)\n\n\ndef build_flash_package(packages_dir, runtime_dir, output_dir, *, allow_build=True):\n    """Standalone wheel preparation. No model Dataset, PDF, or weights are used."""\n    if sys.platform!=\'linux\' or platform.machine()!=\'x86_64\' or sys.version_info[:2]!=(3,13):\n        raise RuntimeError(\'Run this preparation on Kaggle Linux x86_64 / Python 3.13 with an L4 GPU\')\n    import torch\n    if not torch.cuda.is_available():raise RuntimeError(\'Enable a CUDA GPU before building FlashAttention\')\n    output=Path(output_dir);output.mkdir(parents=True,exist_ok=True)\n    root=Path(packages_dir)/\'image_restoration\'\n    plan=json.loads((root/\'manifests/unihir_runtime.json\').read_text(encoding=\'utf8\'))\n    validate_plan(root,plan)\n    if torch.__version__!=plan[\'kernel_reuse\'][\'torch\']:\n        raise RuntimeError(\'Torch differs from the prepared bundle; do not downgrade the kernel\')\n    if len(plan.get(\'native_sources\',[]))!=1:raise ValueError(\'Expected one pinned FlashAttention source\')\n    source=contained(root,plan[\'native_sources\'][0][\'path\'])\n    identity=flash_cache_identity(torch,source)\n    key=hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()\n    folder=Path(runtime_dir)/\'environments\'/(\'flash_attention_\'+key[:16])\n    cache=output/\'flash_attention_cache\'\n    previous={name:os.environ.get(name) for name in (\'VTM_FLASH_CACHE_OUTPUT\',\'VTM_FLASH_REQUIRE_CACHE\')}\n    os.environ[\'VTM_FLASH_CACHE_OUTPUT\']=str(cache)\n    os.environ[\'VTM_FLASH_REQUIRE_CACHE\']=\'0\' if allow_build else \'1\'\n    started=time.monotonic()\n    report=dict(format=\'vtm-flash-attention-package-v1\',identity=identity,status=\'running\',\n        model_weights_loaded=False,model_inference_verified=False,kernel_modified=False)\n    report_path=output/\'flash_attention_build_report.json\'\n    try:\n        if not find_flash_cache(identity,flash_cache_roots(packages_dir)):\n            try:export_flash_cache(packages_dir,runtime_dir,cache)\n            except FileNotFoundError:pass\n        python=folder/\'bin/python\'\n        env=offline_environment()\n        if not python.is_file():\n            folder.mkdir(parents=True,exist_ok=True)\n            subprocess.run([sys.executable,\'-m\',\'venv\',\'--without-pip\',\'--system-site-packages\',str(folder)],\n                env=env,check=True)\n        # Only the native extension is installed; transformer/model dependencies are unnecessary here.\n        native=build_native(python,source,folder,env,packages_dir=packages_dir)\n        subprocess.run([str(python),\'-c\',FLASH_CUDA_PROBE],env=env,check=True,timeout=120)\n        save_flash_cache(identity,Path(native[\'wheel_path\']),cache)\n        (folder/\'install.json\').write_text(json.dumps({\'native\':[native]},indent=2)+\'\\n\',encoding=\'utf8\')\n        report.update(status=\'native_and_cuda_verified\',native=native,cuda_probe_verified=True,\n            seconds=round(time.monotonic()-started,3))\n        report_path.write_text(json.dumps(report,indent=2)+\'\\n\',encoding=\'utf8\')\n        wheel,receipt=find_flash_cache(identity,[cache])\n        archive=output/\'flash_attention_package.zip\';temporary=archive.with_suffix(\'.zip.tmp\')\n        prefix=\'image_restoration/flash_attention_cache/\'+key+\'/\'\n        # Wheels are already compressed; storing them avoids an expensive second compression pass.\n        with zipfile.ZipFile(temporary,\'w\',compression=zipfile.ZIP_STORED) as package:\n            package.write(wheel,prefix+wheel.name)\n            package.write(wheel.parent/\'flash_wheel.json\',prefix+\'flash_wheel.json\')\n            package.write(report_path,prefix+\'build_report.json\')\n        with zipfile.ZipFile(temporary) as package:\n            if package.testzip() is not None:raise ValueError(\'Exported package ZIP integrity failed\')\n        temporary.replace(archive)\n        return dict(report,archive=str(archive),archive_sha256=sha256(archive),\n                    install_under=\'kaggle_upload/packages\',cache_subdirectory=\'image_restoration/flash_attention_cache\')\n    except Exception as exc:\n        report.update(status=\'failed\',error=str(exc),seconds=round(time.monotonic()-started,3))\n        report_path.write_text(json.dumps(report,indent=2)+\'\\n\',encoding=\'utf8\')\n        for name in (\'build.log\',\'build_config.json\',\'build_progress.json\'):\n            log=folder/\'native_build\'/name\n            if log.is_file():shutil.copyfile(log,output/name)\n        raise\n    finally:\n        for name,value in previous.items():\n            if value is None:os.environ.pop(name,None)\n            else:os.environ[name]=value\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--packages-dir\', type=Path, required=True)\n    parser.add_argument(\'--runtime-dir\', type=Path, default=Path(\'/tmp/vtm_image_runtime\'))\n    parser.add_argument(\'--export-flash-cache\',action=\'store_true\',\n                        help=\'Export an existing completed runtime wheel, verify CUDA, then exit\')\n    parser.add_argument(\'--cache-output\',type=Path)\n    parser.add_argument(\'--models\', nargs=\'+\', choices=[\'difftsr\', \'unihir_restoration\',\n                        \'docentr_binarization\',\'degan_binarization\',\'degan_deblurring\',\'degan_watermark_removal\'],\n                        default=[\'difftsr\', \'unihir_restoration\'])\n    args = parser.parse_args(argv)\n    if args.export_flash_cache:\n        export_flash_cache(args.packages_dir,args.runtime_dir,args.cache_output)\n        return 0\n    result = {model: prepare_runtime(args.packages_dir, args.runtime_dir, model) for model in args.models}\n    print(\'RUNTIME_PYTHONS = \'+repr(result))\n    return 0\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main())\n'
FLASH_RUNTIME_SHA256 = 'b4b279da0f90569521264d82105c361973ad2c7ebd752bd783a4a1bdd4ebe4cf'
if hashlib.sha256(FLASH_RUNTIME_SOURCE.encode('utf8')).hexdigest()!=FLASH_RUNTIME_SHA256:
    raise ValueError('Embedded runtime checksum mismatch')
helper=Path(RUNTIME_DIR)/'bootstrap'/('flash_runtime_'+FLASH_RUNTIME_SHA256[:16]+'.py')
helper.parent.mkdir(parents=True,exist_ok=True)
helper.write_text(FLASH_RUNTIME_SOURCE,encoding='utf8')
spec=importlib.util.spec_from_file_location('vtm_flash_package_runtime',helper)
FLASH_RUNTIME=importlib.util.module_from_spec(spec)
spec.loader.exec_module(FLASH_RUNTIME)

OCR_INSTALLER_SOURCE = '"""Install the prepared Dataset with pip --no-index into isolated /tmp runtimes.\n\nRun with Kaggle\'s Python 3.13 on Linux. No network, Docker, WSL, or OCR inference.\nTongGu\'s mmcv extension is compiled locally against its own Python/Torch/CUDA.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport ast\nimport base64\nimport csv\nfrom email.parser import Parser\nimport gzip\nimport hashlib\nimport importlib.metadata as metadata\nimport importlib.util\nimport io\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport pprint\nimport re\nimport shlex\nimport shutil\nimport subprocess\nimport sys\nimport tarfile\nimport tempfile\nimport zipfile\n\n\n# The original report contained duplicate OS metadata after the active wheel.\n# Its successful torch_numpy_bridge probe explicitly reported NumPy 2.1.3.\n# Correct that known snapshot only; arbitrary core-version changes still fail.\nKERNEL_REPORT_CORRECTIONS = {\n    \'1e47c86c9fe706d2e297ea8c0f574716c1d1aadb7366e8de99b195c720c0c746\': {\n        \'numpy\': {\'reported\': \'1.26.4\', \'active\': \'2.1.3\'},\n    },\n}\nOPENCV_PACKAGES = (\'opencv-python\', \'opencv-python-headless\',\n                   \'opencv-contrib-python\', \'opencv-contrib-python-headless\')\n\n# Normal site/venv initialization, with host instrumentation disabled only in\n# the new process. No kernel files or parent-process import state are changed.\nPRIVATE_SITE_INITIALIZATION = """import sys, site, types\nsys.modules[\'sitecustomize\'] = types.ModuleType(\'sitecustomize\')\nsite.main()\n"""\nPRIVATE_PYTHON_BOOTSTRAP = PRIVATE_SITE_INITIALIZATION + """\nimport os, runpy\nsys.executable = sys.argv.pop(1)\nargs = sys.argv[1:]\nwhile args and args[0] in (\'-u\', \'-I\', \'-S\', \'-E\', \'-s\'):\n    args.pop(0)\nif not args:\n    raise SystemExit(\'Private Python expects -c, -m, or a script path\')\nmode = args.pop(0)\nif mode in (\'--version\', \'-V\'):\n    print(\'Python \' + sys.version.split()[0])\nelif mode == \'-c\':\n    code = args.pop(0)\n    sys.argv = [\'-c\', *args]\n    main = types.ModuleType(\'__main__\')\n    main.__dict__.update(__package__=None, __spec__=None)\n    sys.modules[\'__main__\'] = main\n    exec(compile(code, \'<private-python>\', \'exec\'), main.__dict__)\nelif mode == \'-m\':\n    module = args.pop(0)\n    sys.argv = [module, *args]\n    runpy.run_module(module, run_name=\'__main__\', alter_sys=True)\nelse:\n    sys.argv = [mode, *args]\n    sys.path.insert(0, os.path.dirname(os.path.abspath(mode)))\n    runpy.run_path(mode, run_name=\'__main__\')\n"""\n\n\ndef isolated_python_command(python, args):\n    return [python, \'-u\', \'-I\', \'-S\', \'-c\', PRIVATE_PYTHON_BOOTSTRAP, python, *args]\n\n\ndef host_pip_command(args):\n    return isolated_python_command(sys.executable, [\'-m\', \'pip\', *args])\n\n\ndef command_description(args):\n    return \' \'.join(\'<private Python bootstrap>\' if str(arg) == PRIVATE_PYTHON_BOOTSTRAP else str(arg) for arg in args)\n\n\ndef sha256(path):\n    hasher = hashlib.sha256()\n    with path.open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            hasher.update(chunk)\n    return hasher.hexdigest()\n\n\ndef checked_path(root, relative):\n    path = (root / relative).resolve()\n    if not path.is_relative_to(root.resolve()):\n        raise ValueError(f"Path outside bundle: {relative}")\n    return path\n\n\ndef source_tree_digest(files):\n    data = json.dumps(sorted(files, key=lambda r: r[\'path\']), sort_keys=True, separators=(\',\', \':\')).encode()\n    return hashlib.sha256(data).hexdigest()\n\n\nclass SourceTreeError(ValueError):\n    def __init__(self, message, report):\n        self.report = report\n        missing = report.get(\'unresolved_nonempty\', [])\n        changed = report.get(\'changed\', [])\n        details = [f"{report[\'source\']}: expected={report[\'expected_files\']}, actual={report[\'actual_files\']}"]\n        if missing:\n            details.append(f"Missing nonempty files ({len(missing)}): " + \', \'.join(missing[:12]))\n        if changed:\n            details.append(f"Integrity failure ({len(changed)}): " + \', \'.join(r[\'path\'] for r in changed[:12]))\n        super().__init__(message + \'\\n\' + \'\\n\'.join(details))\n\n\ndef resolve_source_tree(bundle, entry):\n    """Resolve only content verified by the manifest; never write to Kaggle Input."""\n    directory = checked_path(bundle, entry[\'path\'])\n    files = entry[\'files\']\n    expected = {r[\'path\'] for r in files}\n    if len(files) != len(expected) or source_tree_digest(files) != entry[\'sha256\']:\n        raise ValueError(f"Invalid source tree manifest: {entry[\'path\']}")\n    actual = {p.relative_to(directory).as_posix(): p for p in directory.rglob(\'*\') if p.is_file()}\n    folded = {}\n    for name in actual:\n        folded.setdefault(name.casefold(), []).append(name)\n    report = {\'source\': entry[\'path\'], \'directory\': str(directory), \'expected_files\': len(expected),\n              \'actual_files\': len(actual), \'expected_bytes\': sum(r[\'bytes\'] for r in files),\n              \'declared_bytes\': entry[\'bytes\'], \'missing_uploaded\': sorted(expected - actual.keys()),\n              \'ignored_extras\': sorted(actual.keys() - expected), \'recreated_empty\': [],\n              \'resolved_names\': [], \'reused_payloads\': [], \'omitted_bytecode\': [],\n              \'unresolved_nonempty\': [], \'changed\': []}\n    if report[\'expected_bytes\'] != entry[\'bytes\']:\n        raise SourceTreeError(\'Invalid source tree byte count\', report)\n    resolved, verified, cache, pending = {}, {}, {}, []\n\n    def matches(name, record):\n        path = checked_path(directory, name)\n        if path.stat().st_size != record[\'bytes\']:\n            return False\n        if path not in cache:\n            cache[path] = sha256(path)\n        return cache[path] == record[\'sha256\']\n\n    for record in files:\n        name = record[\'path\']\n        checked_path(directory, name)\n        if name in actual:\n            if not matches(name, record):\n                report[\'changed\'].append({\'path\': name, \'expected_bytes\': record[\'bytes\'],\n                                          \'actual_bytes\': actual[name].stat().st_size})\n                continue\n            chosen = name\n        else:\n            candidates = list(folded.get(name.casefold(), []))\n            if record.get(\'restore_path\') and not record.get(\'restore_as_gzip\'):\n                candidates.extend(folded.get(record[\'restore_path\'].casefold(), []))\n            chosen = next((candidate for candidate in candidates if matches(candidate, record)), None)\n            if chosen is None:\n                if record[\'bytes\'] == 0 and record[\'sha256\'] == hashlib.sha256(b\'\').hexdigest():\n                    resolved[name] = None  # Recreate the exact empty payload in /tmp.\n                    report[\'recreated_empty\'].append(name)\n                else:\n                    pending.append(record)\n                continue\n            report[\'resolved_names\'].append({\'expected\': name, \'actual\': chosen})\n        resolved[name] = checked_path(directory, chosen)\n        verified[(record[\'bytes\'], record[\'sha256\'])] = resolved[name]\n    unavailable = []\n    for record in pending:\n        donor = verified.get((record[\'bytes\'], record[\'sha256\']))\n        if donor is None:\n            unavailable.append(record)\n        else:\n            resolved[record[\'path\']] = donor\n            report[\'reused_payloads\'].append({\'expected\': record[\'path\'], \'actual\': donor.relative_to(directory).as_posix()})\n    logical_records = {r.get(\'restore_path\', r[\'path\']): r for r in files}\n    for record in unavailable:\n        source_record = None\n        if (entry.get(\'package\') == \'python-runtime\' and Path(record[\'path\']).suffix == \'.pyc\'\n                and not record.get(\'restore_as_gzip\')):\n            try:\n                source_name = Path(importlib.util.source_from_cache(\n                    record.get(\'restore_path\', record[\'path\']))).as_posix()\n            except ValueError:\n                pass  # Sourceless bytecode remains required.\n            else:\n                source_record = logical_records.get(source_name)\n        if (source_record is not None and source_record[\'path\'] in resolved\n                and not source_record.get(\'restore_as_gzip\')):\n            # Python compiles the verified .py in the writable runtime as needed.\n            report[\'omitted_bytecode\'].append({\'path\': record[\'path\'], \'source\': source_record[\'path\']})\n        else:\n            report[\'unresolved_nonempty\'].append(record[\'path\'])\n    if report[\'unresolved_nonempty\'] or report[\'changed\']:\n        raise SourceTreeError(\'Incomplete or changed source tree\', report)\n    return resolved, report\n\n\ndef verify_source_tree(bundle, entry):\n    _, report = resolve_source_tree(bundle, entry)\n    for key, label in ((\'recreated_empty\', \'file rỗng sẽ tạo lại trong /tmp\'),\n                       (\'resolved_names\', \'tên file được đối chiếu bằng SHA-256\'),\n                       (\'reused_payloads\', \'payload trùng checksum được dùng lại\'),\n                       (\'omitted_bytecode\', \'cache .pyc thiếu; dùng file .py đã xác minh\'),\n                       (\'ignored_extras\', \'file thừa không được chép vào runtime\')):\n        if report[key]:\n            print(f"{entry[\'path\']}: {len(report[key])} {label}.", flush=True)\n    return report\n\n\ndef copy_source_tree(bundle, entry, target):\n    """Copy plain uploaded files into writable storage and restore Linux modes."""\n    resolved, report = resolve_source_tree(bundle, entry)\n    omitted = {r[\'path\'] for r in report[\'omitted_bytecode\']}\n    target = Path(target)\n    target.mkdir(parents=True, exist_ok=True)\n    for record in entry[\'files\']:\n        if record[\'path\'] in omitted:\n            continue\n        source = resolved[record[\'path\']]\n        destination = checked_path(target, record.get(\'restore_path\', record[\'path\']))\n        destination.parent.mkdir(parents=True, exist_ok=True)\n        if source is None:\n            if record.get(\'restore_as_gzip\'):\n                with gzip.open(destination, \'wb\'):\n                    pass\n            else:\n                destination.write_bytes(b\'\')\n        elif record.get(\'restore_as_gzip\'):\n            with source.open(\'rb\') as stream, gzip.open(destination, \'wb\') as output:\n                shutil.copyfileobj(stream, output)\n        else:\n            shutil.copyfile(source, destination)\n        destination.chmod(record[\'mode\'])\n    return target\n\n\ndef prepare_python_runtime(bundle, entry, directory):\n    if entry.get(\'layout\') == \'expanded_directory\':\n        copy_source_tree(bundle, entry, directory / \'python\')\n    else:\n        extract_tar(checked_path(bundle, entry[\'path\']), directory)\n\n\ndef prepare_build_source(bundle, entry, runtime):\n    if entry.get(\'layout\') == \'expanded_directory\':\n        return copy_source_tree(bundle, entry, runtime / \'build_sources\' / Path(entry[\'path\']).name)\n    return checked_path(bundle, entry[\'path\'])\n\n\ndef prepare_tonggu_checkout(bundle, entry, runtime):\n    source_root = runtime / \'tonggu_source\'\n    if entry.get(\'layout\') == \'expanded_directory\':\n        return copy_source_tree(bundle, entry, source_root / \'TongGuOCR\')\n    source_root.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(checked_path(bundle, entry[\'path\'])) as archive:\n        for info in archive.infolist():\n            checked_path(source_root, info.filename)\n        archive.extractall(source_root)\n    return next(source_root.glob(\'TongGuOCR-*/tongguocr/pipeline.py\')).parents[1]\n\n\ndef verify(bundle, *, report_path=None):\n    records = {}\n    reports = []\n    groups = ["paddle", "tonggu", "system_assets"]\n    if (bundle / "app_manifest.json").is_file():\n        groups.append("app")\n    for name in groups:\n        path = bundle / f"{name}_manifest.json"\n        if not path.is_file():\n            raise FileNotFoundError(f"Incomplete bundle: {path}")\n        manifest = json.loads(path.read_text(encoding="utf-8"))\n        if manifest.get("failed"):\n            raise ValueError(f"Bundle has failed downloads: {name}")\n        if name == \'tonggu\' and any(r[\'package\'].lower().replace(\'_\', \'-\') == \'nvidia-nccl-cu13\'\n                                    for r in manifest.get(\'reused\', [])):\n            raise ValueError(\'Dataset packages còn dùng lại NCCL từ kernel. Cập nhật Dataset packages \'\n                             \'bằng bộ đã bổ sung wheel NCCL riêng cho TongGu, rồi gắn phiên bản mới \'\n                             \'vào notebook. Chỉ thay ô setup chưa đủ cho lỗi checksum NCCL.\')\n        for entry in manifest["records"]:\n            if entry.get(\'layout\') == \'expanded_directory\':\n                try:\n                    reports.append(verify_source_tree(bundle, entry))\n                except SourceTreeError as exc:\n                    if report_path is not None:\n                        report_path = Path(report_path)\n                        report_path.parent.mkdir(parents=True, exist_ok=True)\n                        report_path.write_text(json.dumps({\'ok\': False, \'source_trees\': reports + [exc.report]},\n                            ensure_ascii=False, indent=2), encoding=\'utf-8\')\n                    raise\n                records[entry[\'path\']] = entry\n                continue\n            path = checked_path(bundle, entry["path"])\n            if path.stat().st_size != entry["bytes"] or sha256(path) != entry["sha256"]:\n                raise ValueError(f"Integrity failure: {path.name}")\n            records[entry["path"]] = entry\n    if report_path is not None:\n        report_path = Path(report_path)\n        report_path.parent.mkdir(parents=True, exist_ok=True)\n        report_path.write_text(json.dumps({\'ok\': True, \'source_trees\': reports}, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    print(f"Verified {len(records)} artifacts, {sum(r[\'bytes\'] for r in records.values()) / 1024**3:.2f} GiB", flush=True)\n    return list(records.values())\n\n\ndef run(args, environment=None, *, cwd=None):\n    print("RUN:", command_description(args), flush=True)\n    subprocess.run(list(map(str, args)), env=environment, cwd=cwd, check=True)\n\n\ndef extract_tar(path, target):\n    target.mkdir(parents=True, exist_ok=True)\n    with tarfile.open(path) as archive:\n        archive.extractall(target, filter="data")\n\n\ndef target_site(python, runtime):\n    code = PRIVATE_SITE_INITIALIZATION + "\\nimport json,sysconfig; print(json.dumps([sysconfig.get_paths()[\'purelib\'],sys.version.split()[0]]))"\n    site, version = json.loads(subprocess.check_output([str(python), "-u", "-I", "-S", "-c", code],\n                                                    env=native_environment(), text=True).strip())\n    site = Path(site).resolve()\n    if not site.is_relative_to(runtime.resolve()):\n        raise ValueError(f"Target interpreter is outside the private runtime: {site}")\n    site.mkdir(parents=True, exist_ok=True)\n    return site, version\n\n\ndef clear_kernel_links(site):\n    ledger = site / ".vtm_kernel_links.json"\n    if not ledger.is_file():\n        return\n    for relative in json.loads(ledger.read_text(encoding="utf-8")):\n        path = site / relative\n        # Check the lexical parent; resolving a legitimate symlink points at the kernel.\n        if not path.parent.resolve().is_relative_to(site.resolve()):\n            raise ValueError(f"Kernel link escapes target site: {relative}")\n        if path.is_symlink():\n            path.unlink()\n    ledger.unlink()\n\n\ndef clear_reused_packages(site, records):\n    """Remove previous private copies, including a failed install; never follow kernel links."""\n    from packaging.utils import canonicalize_name\n    names = {canonicalize_name(r[\'package\']) for r in records}\n    owned, protected = set(), set()\n    for dist in metadata.distributions(path=[str(site)]):\n        name = canonicalize_name(dist.metadata.get(\'Name\', \'\'))\n        paths = owned if name in names else protected\n        for row in csv.reader(io.StringIO(dist.read_text(\'RECORD\') or \'\')):\n            if row:\n                paths.add(row[0])\n    ledger = site / \'.vtm_kernel_files.json\'\n    if ledger.is_file():\n        owned.update(json.loads(ledger.read_text(encoding=\'utf-8\')))\n    targets = []\n    for relative in owned - protected:\n        path = Path(relative)\n        if path.is_absolute() or \'..\' in path.parts:\n            continue  # Installed console scripts are outside the private site.\n        target = site / path\n        if not target.parent.resolve().is_relative_to(site.resolve()):\n            raise ValueError(f\'Reused package cleanup escapes private site: {relative}\')\n        if target.is_symlink() or target.is_file():\n            targets.append(target)\n    for target in targets:\n        target.unlink()\n    if ledger.is_file():\n        ledger.unlink()\n    clear_kernel_links(site)\n\n\ndef private_kernel_cleanup_records(manifest):\n    """Clean kernel copies; let the dedicated cv2 repair own its shared namespace."""\n    from packaging.utils import canonicalize_name\n    records = [*manifest.get(\'reused\', []), *manifest.get(\'kernel_overlap_repairs\', [])]\n    shipped = {canonicalize_name(r[\'package\']) for r in manifest.get(\'records\', [])}\n    if shipped.intersection(OPENCV_PACKAGES):\n        records = [r for r in records if canonicalize_name(r[\'package\']) not in OPENCV_PACKAGES]\n    return list({canonicalize_name(r[\'package\']): r for r in records}.values())\n\n\ndef opencv_metadata_directories(site):\n    """Find even damaged OpenCV metadata without relying on importer caches."""\n    from packaging.utils import canonicalize_name\n    return [path for path in Path(site).glob(\'*.dist-info\')\n            if canonicalize_name(path.name[:-len(\'.dist-info\')].rsplit(\'-\', 1)[0]) in OPENCV_PACKAGES]\n\n\ndef clear_private_opencv_files(site, runtime):\n    """Remove only OpenCV-owned namespaces in a verified private site."""\n    site, runtime = Path(site).resolve(), Path(runtime).resolve()\n    if site == runtime or not site.is_relative_to(runtime):\n        raise ValueError(f\'OpenCV cleanup site is outside private runtime: {site}\')\n    targets = [site / \'cv2\', site / \'cv2.py\', *site.glob(\'cv2.*.so\'), *site.glob(\'cv2*.pyd\'),\n               *opencv_metadata_directories(site)]\n    targets.extend(site / (name.replace(\'-\', \'_\') + \'.libs\') for name in OPENCV_PACKAGES)\n    for target in targets:\n        if target.parent.resolve() != site:\n            raise ValueError(f\'OpenCV cleanup target escapes private site: {target}\')\n        if target.is_symlink():\n            target.unlink()  # Never follow a kernel payload link.\n        elif target.exists():\n            resolved = target.resolve()\n            if resolved == site or not resolved.is_relative_to(site):\n                raise ValueError(f\'OpenCV cleanup target escapes private site: {target}\')\n            if target.is_dir():\n                shutil.rmtree(target)\n            else:\n                target.unlink()\n\n\ndef audit_private_opencv(wheel, site):\n    """Compare the installed cv2 payload and vendor libraries against wheel RECORD."""\n    site = Path(site).resolve()\n    problems, checked = [], 0\n    with zipfile.ZipFile(wheel) as archive:\n        record_name = next(n for n in archive.namelist() if n.endswith(\'.dist-info/RECORD\') and n.count(\'/\') == 1)\n        expected_info = record_name.rsplit(\'/\', 1)[0]\n        expected_files = set(archive.namelist())\n        installed_record = site / record_name\n        if installed_record.is_symlink() or not installed_record.is_file():\n            problems.append({\'path\': record_name, \'reason\': \'missing_record\'})\n        for row in csv.reader(io.StringIO(archive.read(record_name).decode(\'utf-8\'))):\n            if not row or row[0] == record_name:\n                continue  # pip regenerates RECORD with its own install bookkeeping.\n            name, checksum, size = row\n            relative = Path(name)\n            if relative.is_absolute() or \'..\' in relative.parts or not checksum:\n                raise ValueError(f\'Unsafe or unverifiable OpenCV wheel entry: {name}\')\n            target = site / relative\n            if not target.parent.resolve().is_relative_to(site):\n                problems.append({\'path\': name, \'reason\': \'parent_symlink\'})\n                continue\n            if target.is_symlink() or not target.is_file():\n                problems.append({\'path\': name, \'reason\': \'linked_or_missing\'})\n                continue\n            algorithm, expected_hash = checksum.split(\'=\', 1)\n            if size and target.stat().st_size != int(size):\n                problems.append({\'path\': name, \'reason\': \'size\'})\n                continue\n            with target.open(\'rb\') as stream:\n                actual = base64.urlsafe_b64encode(hashlib.file_digest(stream, algorithm).digest()).rstrip(b\'=\').decode()\n            if actual != expected_hash:\n                problems.append({\'path\': name, \'reason\': \'checksum\'})\n            checked += 1\n        for metadata_dir in opencv_metadata_directories(site):\n            if metadata_dir.name != expected_info and (metadata_dir.is_file() or any(metadata_dir.iterdir())):\n                problems.append({\'path\': metadata_dir.name, \'reason\': \'other_provider\'})\n        cv2_root = site / \'cv2\'\n        if cv2_root.is_dir() and cv2_root.resolve().is_relative_to(site):\n            for path in cv2_root.rglob(\'*\'):\n                name = path.relative_to(site).as_posix()\n                if (path.is_file() and path.suffix in {\'.py\', \'.so\', \'.pyd\'}\n                        and name not in expected_files and \'__pycache__\' not in path.parts):\n                    problems.append({\'path\': name, \'reason\': \'untracked_loader_or_binary\'})\n        for path in [site / \'cv2.py\', *site.glob(\'cv2.*.so\'), *site.glob(\'cv2*.pyd\')]:\n            if path.is_file() and path.name not in expected_files:\n                problems.append({\'path\': path.name, \'reason\': \'legacy_root_module\'})\n    return {\'wheel\': str(wheel), \'site\': str(site), \'checked_files\': checked, \'problems\': problems}\n\n\nPRIVATE_OPENCV_PROBE = """import importlib.metadata as md, json, sys\nfrom pathlib import Path\nsite, expected, version, output = sys.argv[1:5]\nreport = {\'package\': expected, \'version\': version}\ntry:\n    import cv2, numpy as np\n    report.update(origin=getattr(cv2,\'__file__\',None), cv2_version=getattr(cv2,\'__version__\',None))\n    needed = [\'resize\',\'cvtColor\',\'warpAffine\',\'getRotationMatrix2D\',\'findContours\',\'threshold\',\n              \'minAreaRect\',\'boxPoints\',\'copyMakeBorder\',\'imread\',\'imwrite\',\'imencode\',\'imdecode\']\n    missing = [name for name in needed if not callable(getattr(cv2,name,None))]\n    report[\'missing_api\'] = missing\n    assert not missing, \'Incomplete cv2 API: \' + \', \'.join(missing)\n    assert report[\'origin\'] and Path(report[\'origin\']).resolve().is_relative_to(Path(site).resolve()), report\n    variants = {\'opencv-python\',\'opencv-python-headless\',\'opencv-contrib-python\',\'opencv-contrib-python-headless\'}\n    providers = {d.metadata.get(\'Name\',\'\').lower().replace(\'_\',\'-\'): d.version\n                 for d in md.distributions(path=[site]) if d.metadata.get(\'Name\',\'\').lower().replace(\'_\',\'-\') in variants}\n    report[\'providers\'] = providers\n    assert providers == {expected:version}, providers\n    image = np.zeros((8,8,3),dtype=np.uint8); image[2:6,2:6] = 255\n    assert cv2.resize(image,(16,16)).shape == (16,16,3)\n    gray = cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)\n    _, binary = cv2.threshold(gray,0,255,cv2.THRESH_BINARY | cv2.THRESH_OTSU)\n    contours, _ = cv2.findContours(binary,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)\n    assert contours and cv2.boxPoints(cv2.minAreaRect(contours[0])).shape == (4,2)\n    matrix = cv2.getRotationMatrix2D((4.,4.),0.,1.)\n    assert cv2.warpAffine(image,matrix,(8,8)).shape == image.shape\n    assert cv2.copyMakeBorder(image,1,1,1,1,cv2.BORDER_CONSTANT).shape == (10,10,3)\n    ok, encoded = cv2.imencode(\'.png\',image)\n    assert ok and cv2.imdecode(encoded,cv2.IMREAD_COLOR).shape == image.shape\n    report[\'ok\'] = True\n    print(json.dumps(report))\nexcept Exception as exc:\n    report.update(ok=False,error=repr(exc),sys_path=sys.path)\n    raise\nfinally:\n    Path(output).write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n"""\n\n\ndef ensure_private_opencv(bundle, manifest, site, python, environment, runtime):\n    """Repair partial OpenCV installs even when pip considers their metadata satisfied."""\n    from packaging.utils import canonicalize_name\n    records = [r for r in manifest.get(\'records\', []) if canonicalize_name(r[\'package\']) in OPENCV_PACKAGES]\n    if not records:\n        return None\n    if len(records) != 1:\n        raise ValueError(\'A private runtime must ship exactly one OpenCV provider\')\n    record = records[0]\n    wheel = checked_path(Path(bundle), record[\'path\'])\n    if sha256(wheel) != record[\'sha256\']:\n        raise ValueError(f\'OpenCV repair wheel checksum differs: {wheel}\')\n    site, runtime = Path(site).resolve(), Path(runtime).resolve()\n    if site == runtime or not site.is_relative_to(runtime):\n        raise ValueError(f\'OpenCV site is outside private runtime: {site}\')\n    audit = audit_private_opencv(wheel, site)\n    repaired = bool(audit[\'problems\'])\n    if repaired:\n        print(f\'Repair incomplete OpenCV payload: {len(audit["problems"])} mismatches; \'\n              f\'first: {audit["problems"][:5]}\', flush=True)\n        clear_private_opencv_files(site, runtime)\n        run(host_pip_command([\'--python\', python, \'install\', \'--no-index\', \'--no-deps\',\n                              \'--force-reinstall\', wheel]), environment)\n        audit = audit_private_opencv(wheel, site)\n    audit[\'repaired\'] = repaired\n    audit_path = site / \'.vtm_opencv_payload.json\'\n    audit_path.write_text(json.dumps(audit, indent=2), encoding=\'utf-8\')\n    if audit[\'problems\']:\n        raise RuntimeError(f\'OpenCV payload remains incomplete after reinstall: {audit_path}\')\n    probe = site / \'.vtm_opencv_probe.json\'\n    run([python, \'-c\', PRIVATE_OPENCV_PROBE, site, canonicalize_name(record[\'package\']), record[\'version\'], probe], environment)\n    print(f\'OpenCV payload verified: {audit["checked_files"]} files; probe: {probe}\', flush=True)\n    return str(probe)\n\n\ndef kernel_distribution_compatible(dist, name, target_version):\n    from packaging.specifiers import SpecifierSet\n    from packaging.tags import compatible_tags, cpython_tags, parse_tag, platform_tags\n    requires_python = dist.metadata.get(\'Requires-Python\')\n    if requires_python and not SpecifierSet(requires_python).contains(target_version):\n        raise ValueError(f\'Kernel package requires another Python: {name} {requires_python}, target {target_version}\')\n    minor = int(target_version.split(\'.\')[1])\n    platforms = list(platform_tags())\n    tags = set(cpython_tags((3, minor), abis=[f\'cp3{minor}\'], platforms=platforms))\n    tags.update(compatible_tags((3, minor), interpreter=f\'cp3{minor}\', platforms=platforms))\n    wheel = dist.read_text(\'WHEEL\') or \'\'\n    wheel_tags = set().union(*(parse_tag(line[5:].strip()) for line in wheel.splitlines() if line.startswith(\'Tag: \')))\n    if (wheel_tags or minor != sys.version_info.minor) and not wheel_tags & tags:\n        raise ValueError(f\'Kernel package ABI cannot be reused in Python {target_version}: {name}\')\n\n\ndef plan_kernel_reuse(bundle, manifest, target_version, report_path):\n    """Check live metadata and its dependency closure against the fixed offline artifacts."""\n    from packaging.markers import default_environment\n    from packaging.requirements import Requirement\n    from packaging.utils import canonicalize_name\n    from packaging.version import Version\n    environment = {**default_environment(), \'python_version\': \'.\'.join(target_version.split(\'.\')[:2]),\n                   \'python_full_version\': target_version, \'os_name\': \'posix\', \'sys_platform\': \'linux\',\n                   \'platform_system\': \'Linux\', \'platform_machine\': \'x86_64\', \'implementation_name\': \'cpython\'}\n    selected, live, original = {}, {}, {canonicalize_name(r[\'package\']): r for r in manifest.get(\'reused\', [])}\n    extras = {canonicalize_name(n): set(values) for n, values in manifest.get(\'extras\', {}).items()}\n    report = {\'target_python\': target_version, \'version_changes\': [], \'added_dependencies\': [],\n              \'excluded_packages\': [], \'errors\': []}\n    protected = {\'torch\', \'torchvision\', \'transformers\', \'numpy\', \'paddlepaddle-gpu\', \'paddleocr\', \'paddlex\'}\n    corrections = KERNEL_REPORT_CORRECTIONS.get(manifest.get(\'base_report_sha256\'), {})\n\n    def select_kernel(name):\n        if name in OPENCV_PACKAGES:\n            existing = set(selected).intersection(OPENCV_PACKAGES) - {name}\n            if existing:\n                raise ValueError(f\'Multiple OpenCV providers for cv2: {name} conflicts with {sorted(existing)}\')\n        try:\n            dist = metadata.distribution(name)\n        except metadata.PackageNotFoundError as exc:\n            raise ValueError(f\'Offline dependency unavailable in kernel: {name}\') from exc\n        if canonicalize_name(dist.metadata.get(\'Name\', \'\')) != name:\n            raise ValueError(f\'Kernel distribution identity differs: {name}\')\n        kernel_distribution_compatible(dist, name, target_version)\n        prior = original.get(name)\n        baseline = prior[\'version\'] if prior else None\n        correction = corrections.get(name, {})\n        if baseline == correction.get(\'reported\'):\n            baseline = correction.get(\'active\', baseline)\n        if prior and (name in protected or name.startswith(\'nvidia-\')) and Version(dist.version) != Version(baseline):\n            raise ValueError(f\'Core kernel package changed: {name} expected {baseline}, got {dist.version}\')\n        if prior and Version(dist.version) != Version(prior[\'version\']):\n            report[\'version_changes\'].append({\'package\': name, \'reported\': prior[\'version\'], \'actual\': dist.version})\n        if prior is None:\n            report[\'added_dependencies\'].append({\'package\': name, \'version\': dist.version})\n        live[name] = {\'package\': name, \'version\': dist.version,\n                      \'reported_version\': prior[\'version\'] if prior else None}\n        selected[name] = {\'version\': dist.version, \'requires\': dist.metadata.get_all(\'Requires-Dist\', [])}\n\n    try:\n        for record in manifest[\'records\']:\n            name = canonicalize_name(record[\'package\'])\n            requires = record.get(\'requires\', [])\n            if record[\'kind\'] == \'wheel\':\n                with zipfile.ZipFile(checked_path(bundle, record[\'path\'])) as archive:\n                    candidates = [n for n in archive.namelist() if n.endswith(\'.dist-info/METADATA\') and n.count(\'/\') == 1]\n                    if len(candidates) != 1:\n                        raise ValueError(f\'Invalid wheel metadata: {record["path"]}\')\n                    data = Parser().parsestr(archive.read(candidates[0]).decode(\'utf-8\'))\n                if canonicalize_name(data[\'Name\']) != name or Version(data[\'Version\']) != Version(record[\'version\']):\n                    raise ValueError(f\'Offline wheel identity differs: {record["path"]}\')\n                requires = data.get_all(\'Requires-Dist\', [])\n            selected[name] = {\'version\': record[\'version\'], \'requires\': requires}\n        shipped_opencv = set(selected).intersection(OPENCV_PACKAGES)\n        if len(shipped_opencv) > 1:\n            raise ValueError(f\'Multiple shipped OpenCV providers for cv2: {sorted(shipped_opencv)}\')\n        for name in original:\n            if name in selected:\n                raise ValueError(f\'Both shipped and reused package: {name}\')\n            if shipped_opencv and name in OPENCV_PACKAGES:\n                report[\'excluded_packages\'].append({\'package\': name, \'reason\':\n                    f\'cv2 is supplied by the shipped {next(iter(shipped_opencv))} wheel\'})\n                continue\n            select_kernel(name)\n        # Iterate to include live dependencies introduced by newer auxiliary packages,\n        # and extras activated by dependencies discovered later in the traversal.\n        requirements = [("root", Requirement(text)) for text in manifest.get(\'roots\', [])]\n        changed = True\n        while changed:\n            changed = False\n            for name, record in list(selected.items()):\n                for text in record[\'requires\']:\n                    req = Requirement(text)\n                    if req.marker and not any(req.marker.evaluate({**environment, \'extra\': extra})\n                                              for extra in {\'\', *extras.get(name, set())}):\n                        continue\n                    if (name, req) not in requirements:\n                        requirements.append((name, req))\n                        changed = True\n            for parent, req in requirements:\n                if parent == \'root\' and req.marker and not req.marker.evaluate({**environment, \'extra\': \'\'}):\n                    continue\n                if req.url:\n                    raise ValueError(f\'Offline URL dependency is unsupported: {parent} -> {req}\')\n                name = canonicalize_name(req.name)\n                if name not in selected:\n                    try:\n                        select_kernel(name)\n                    except ValueError as exc:\n                        raise ValueError(f\'{parent} -> {req}: {exc}\') from exc\n                    changed = True\n                if not req.specifier.contains(selected[name][\'version\'], prereleases=True):\n                    raise ValueError(f\'Unsatisfied offline dependency: {parent} -> {req}; have {selected[name]["version"]}\')\n                active = extras.setdefault(name, set())\n                if not req.extras <= active:\n                    active.update(req.extras)\n                    changed = True\n        report.update(ok=True, reused=list(live.values()), dependency_edges_checked=len(requirements),\n                      opencv_provider=next(iter(set(selected).intersection(OPENCV_PACKAGES)), None))\n    except Exception as exc:\n        report.update(ok=False)\n        report[\'errors\'].append(str(exc))\n        raise\n    finally:\n        report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    for change in report[\'version_changes\']:\n        print(f"Compatible kernel update: {change[\'package\']} {change[\'reported\']} -> {change[\'actual\']}", flush=True)\n    if report[\'opencv_provider\']:\n        print(f"OpenCV provider: {report[\'opencv_provider\']}; excluded {len(report[\'excluded_packages\'])} kernel variants", flush=True)\n    print(f"Kernel dependency plan: {len(live)} packages, {len(requirements)} dependency constraints checked", flush=True)\n    return list(live.values())\n\n\ndef reuse_kernel_packages(records, site, target_version, *, bundle=None, manifest=None):\n    """Copy Python files and link ABI-compatible native files, never add cp313 site to cp310."""\n    from packaging.version import Version\n    if manifest is not None:\n        records = plan_kernel_reuse(bundle, manifest, target_version, site / \'.vtm_kernel_plan.json\')\n    links, copied, reused = [], [], []\n    try:\n        for expected in records:\n            dist = metadata.distribution(expected["package"])\n            if Version(dist.version) != Version(expected["version"]):\n                raise ValueError(f"Kernel package changed: {expected[\'package\']} expected {expected[\'version\']}, got {dist.version}")\n            kernel_distribution_compatible(dist, expected[\'package\'], target_version)\n            source_root = Path(dist.locate_file("")).resolve()\n            record_text = dist.read_text("RECORD")\n            if not record_text:\n                raise ValueError(f"Kernel package has no verifiable RECORD: {expected[\'package\']}")\n            count = 0\n            for row in csv.reader(io.StringIO(record_text)):\n                if not row:\n                    continue\n                item, checksum, _size = row\n                relative = Path(item)\n                if relative.is_absolute() or ".." in relative.parts or "__pycache__" in relative.parts or relative.suffix == ".pyc":\n                    continue  # console scripts and interpreter-specific bytecode are not shared\n                source = Path(dist.locate_file(item)).resolve()\n                if not source.is_relative_to(source_root) or not source.is_file():\n                    raise ValueError(f"Kernel package file unavailable: {item}")\n                if checksum:\n                    algorithm, expected_hash = checksum.split("=", 1)\n                    hasher = hashlib.new(algorithm)\n                    with source.open("rb") as stream:\n                        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n                            hasher.update(block)\n                    actual = base64.urlsafe_b64encode(hasher.digest()).rstrip(b"=").decode()\n                    if actual != expected_hash:\n                        raise ValueError(f"Kernel RECORD checksum differs: {expected[\'package\']}/{item}")\n                elif ".so" in source.name:\n                    raise ValueError(f"Unverifiable native kernel file: {item}")\n                target = site / relative\n                if not target.parent.resolve().is_relative_to(site.resolve()):\n                    raise ValueError(f"Reused package file escapes private site: {relative}")\n                target.parent.mkdir(parents=True, exist_ok=True)\n                if target.exists():\n                    if sha256(target) != sha256(source):\n                        raise ValueError(f"Package files overlap with different bytes: {relative}")\n                    continue\n                if ".so" in source.name:\n                    target.symlink_to(source)\n                    links.append(relative.as_posix())\n                else:\n                    shutil.copy2(source, target)\n                copied.append(relative.as_posix())\n                count += 1\n            reused.append({"package": expected["package"], "version": dist.version,\n                           "reported_version": expected.get(\'reported_version\', expected[\'version\']), "files": count})\n    finally:\n        (site / ".vtm_kernel_links.json").write_text(json.dumps(links, indent=2), encoding="utf-8")\n        (site / ".vtm_kernel_files.json").write_text(json.dumps(copied, indent=2), encoding="utf-8")\n    print(f"Reused {len(reused)} kernel distributions in Python {target_version}", flush=True)\n    return reused\n\n\ndef component_manifest(bundle, name):\n    path = bundle / f"{name}_manifest.json"\n    return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else {}\n\n\ndef private_python(python, site):\n    """Use the same private CUDA library search order for probes and OCR workers."""\n    libraries = sorted({str(path.parent) for path in (site / "nvidia").rglob("*.so*") if path.is_file()})\n    wrapper = python.with_name("vtm-python")\n    prefix = shlex.quote(":".join(libraries))\n    script = \'#!/bin/sh\\nunset PYTHONPATH PYTHONHOME PYTHONUSERBASE\\nexport PYTHONNOUSERSITE=1\\nexport MPLBACKEND=Agg\\n\'\n    if libraries:\n        script += \'export LD_LIBRARY_PATH=\' + prefix + \'"${LD_LIBRARY_PATH:+:$LD_LIBRARY_PATH}"\\n\'\n    command = isolated_python_command(python, [])\n    command[-1] = wrapper  # Backend subprocesses must re-enter this launcher too.\n    script += \'exec \' + \' \'.join(shlex.quote(str(arg)) for arg in command) + \' "$@"\\n\'\n    wrapper.write_text(script, encoding="utf-8", newline="\\n")\n    wrapper.chmod(0o755)\n    return wrapper\n\n\ndef native_environment(cuda=None):\n    env = os.environ.copy()\n    for name in (\'PYTHONPATH\', \'PYTHONHOME\', \'PYTHONUSERBASE\', \'_PYTHON_SYSCONFIGDATA_NAME\', \'_PYTHON_HOST_PLATFORM\'):\n        env.pop(name, None)\n    env.update(PIP_NO_INDEX="1", HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",\n               HF_HUB_DISABLE_TELEMETRY="1", PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK="True", PYTHONNOUSERSITE=\'1\',\n               MPLBACKEND=\'Agg\')\n    if cuda:\n        env.update(CUDA_HOME=str(cuda), CUDA_PATH=str(cuda),\n                   MAX_JOBS=os.environ.get(\'MAX_JOBS\', \'2\'))\n        env["PATH"] = str(cuda / "bin") + os.pathsep + env.get("PATH", "")\n        env["LD_LIBRARY_PATH"] = str(cuda / "lib64") + os.pathsep + env.get("LD_LIBRARY_PATH", "")\n    return env\n\n\ndef configure_cuda_architecture(python, environment):\n    """Compile native ops for the GPUs in this session, including L4 SM 8.9."""\n    source = \'\'\'import json,torch\nassert torch.cuda.is_available(), \'Native CUDA build requires a visible GPU\'\nprint(json.dumps(sorted({f\'{major}.{minor}\' for major,minor in\n    [torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())]})))\n\'\'\'\n    output = subprocess.check_output([str(python), \'-c\', source], env=environment, text=True)\n    arches = json.loads(output.splitlines()[-1])\n    if not arches or any(not re.fullmatch(r\'\\d+\\.\\d+\', arch) for arch in arches):\n        raise RuntimeError(\'Cannot determine CUDA architecture for native build\')\n    environment[\'TORCH_CUDA_ARCH_LIST\'] = \';\'.join(arches)\n    return environment[\'TORCH_CUDA_ARCH_LIST\']\n\n\ndef repair_python_build_config(python_root, environment):\n    """Relocate standalone Python\'s build paths in the writable runtime only."""\n    python_root = Path(python_root).resolve()\n    configs = list(python_root.glob(\'lib/python*/_sysconfigdata*.py\'))\n    if len(configs) != 1 or not configs[0].resolve().is_relative_to(python_root):\n        raise ValueError(f\'Cannot identify private Python build config: {python_root}\')\n    path = configs[0]\n    tree = ast.parse(path.read_text(encoding=\'utf-8\'))\n    values = next(ast.literal_eval(n.value) for n in tree.body if isinstance(n, ast.Assign)\n                  and any(isinstance(t, ast.Name) and t.id == \'build_time_vars\' for t in n.targets))\n    previous = dict(values)\n    old_prefix = values.get(\'prefix\', \'/install\')\n    if old_prefix != str(python_root):\n        pattern = re.compile(re.escape(old_prefix) + r\'(?=/|$|[\\s:])\')\n        values = {key: pattern.sub(lambda _: str(python_root), value) if isinstance(value, str) else value\n                  for key, value in values.items()}\n    values.update(CC=environment[\'CC\'] + \' -pthread\', CXX=environment[\'CXX\'] + \' -pthread\',\n                  LDSHARED=environment[\'LDSHARED\'], LDCXXSHARED=environment[\'LDCXXSHARED\'],\n                  BLDSHARED=environment[\'LDSHARED\'])\n    changed = sorted(key for key in values if values[key] != previous.get(key))\n    if changed:\n        path.write_text(\'build_time_vars = \' + pprint.pformat(values, sort_dicts=True, width=110) + \'\\n\', encoding=\'utf-8\')\n        # The installer is cp313 but this config belongs to cp310: remove\n        # target-interpreter caches too, even on same-second/same-size edits.\n        for cached in (path.parent / \'__pycache__\').glob(path.stem + \'.*.pyc\'):\n            if cached.is_file():\n                cached.unlink()\n    include = Path(values[\'INCLUDEPY\'])\n    if not include.is_relative_to(python_root) or not (include / \'Python.h\').is_file() or not (include / \'pyconfig.h\').is_file():\n        raise ValueError(f\'Private Python headers unavailable after relocation: {include}\')\n    return {\'python_root\': str(python_root), \'include\': str(include), \'ext_suffix\': values[\'EXT_SUFFIX\'],\n            \'CC\': values[\'CC\'], \'CXX\': values[\'CXX\'], \'changed_keys\': changed}\n\n\ndef prepare_build_toolchain(python, runtime, environment):\n    """Prove that the host compiler can build and import a private Python extension."""\n    env = dict(environment)\n    cc, cxx = (shutil.which(name, path=env.get(\'PATH\')) for name in (\'gcc\', \'g++\'))\n    if not cc or not cxx:\n        raise RuntimeError(\'Native offline builds need gcc and g++; compiler preflight stopped before source installs\')\n    env.update(CC=cc, CXX=cxx, LDSHARED=shlex.quote(cc) + \' -pthread -shared\',\n               LDCXXSHARED=shlex.quote(cxx) + \' -pthread -shared\', NVCC_CCBIN=cxx,\n               MAX_JOBS=env.get(\'MAX_JOBS\', \'2\'), PIP_DISABLE_PIP_VERSION_CHECK=\'1\',\n               PYTHONUNBUFFERED=\'1\')\n    configuration = repair_python_build_config(Path(python).parent.parent, env)\n    for label, compiler in [(\'CC_version\', cc), (\'CXX_version\', cxx)]:\n        configuration[label] = subprocess.check_output([compiler, \'--version\'], env=env, text=True).splitlines()[0]\n        print(configuration[label], flush=True)\n    probe = Path(runtime) / \'build_probe\'\n    probe.mkdir(parents=True, exist_ok=True)\n    source = probe / \'_vtm_build_probe.c\'\n    source.write_text(\'\'\'#include <Python.h>\nstatic PyObject *answer(PyObject *self, PyObject *args) { return PyLong_FromLong(42); }\nstatic PyMethodDef methods[] = {{"answer", answer, METH_NOARGS, "probe"}, {NULL, NULL, 0, NULL}};\nstatic struct PyModuleDef module = {PyModuleDef_HEAD_INIT, "_vtm_build_probe", NULL, -1, methods};\nPyMODINIT_FUNC PyInit__vtm_build_probe(void) { return PyModule_Create(&module); }\n\'\'\', encoding=\'utf-8\')\n    extension = probe / (\'_vtm_build_probe\' + configuration[\'ext_suffix\'])\n    run([cc, \'-shared\', \'-fPIC\', \'-O2\', \'-I\' + configuration[\'include\'], source, \'-o\', extension], env)\n    run(isolated_python_command(python, [\'-c\', "import sys; sys.path.insert(0,sys.argv[1]); import _vtm_build_probe; "\n         "assert _vtm_build_probe.answer()==42; print(\'C extension build/import preflight OK\')", probe]), env)\n    (probe / \'toolchain.json\').write_text(json.dumps(configuration, indent=2), encoding=\'utf-8\')\n    return env, configuration\n\n\ndef patch_mmcv_build_source(source):\n    """Apply narrowly scoped source fixes for the pinned Torch 2.10/CUDA 13 build."""\n    source = Path(source)\n    changed = []\n    for relative, old, new in (\n        (\'setup.py\', \'-std=c++14\', \'-std=c++17\'),\n        (\'mmcv/ops/csrc/pytorch/spconv_utils.h\', \'tensor.type().scalarType()\', \'tensor.scalar_type()\'),\n    ):\n        path = source / relative\n        if not path.is_file():\n            continue\n        text = path.read_text(encoding=\'utf-8\')\n        if old in text:\n            path.write_text(text.replace(old, new), encoding=\'utf-8\')\n            changed.append(relative)\n    return changed\n\n\ndef source_wheel_identity(bundle,record,runtime,environment,profile):\n    def portable(value):\n        if isinstance(value,dict):return {k:portable(v) for k,v in value.items()}\n        if isinstance(value,(list,tuple)):return [portable(v) for v in value]\n        if isinstance(value,str):\n            return value.replace(str(Path(runtime).resolve()),\'<runtime>\').replace(str(Path(bundle).resolve()),\'<bundle>\')\n        return value\n    return portable({\'source_sha256\':record[\'sha256\'],\'package\':record[\'package\'],\n        \'version\':record[\'version\'],\'profile\':profile,\'build_schema\':3,\n        \'platform\':sys.platform,\'machine\':platform.machine(),\'libc\':list(platform.libc_ver()),\n        \'build_environment\':{name:environment.get(name) for name in (\'CC\',\'CXX\',\'LDSHARED\',\n            \'LDCXXSHARED\',\'CFLAGS\',\'CPPFLAGS\',\'CXXFLAGS\',\'LDFLAGS\',\'CUDA_HOME\',\'NVCC_CCBIN\',\n            \'NVCC_FLAGS\',\'MMCV_WITH_OPS\',\'TORCH_CUDA_ARCH_LIST\')}})\n\n\ndef verify_target_wheel_tags(python,wheel):\n    from packaging.utils import parse_wheel_filename\n    code="import json; from packaging.tags import sys_tags; print(json.dumps([str(t) for t in sys_tags()]))"\n    target=set(json.loads(subprocess.check_output([str(python),\'-c\',code],env=native_environment(),text=True)))\n    if not target.intersection(str(t) for t in parse_wheel_filename(wheel.name)[3]):\n        raise ValueError(f\'Native wheel ABI does not match target interpreter: {wheel.name}\')\n\n\ndef build_offline_source_wheel(bundle, record, runtime, python, environment, profile):\n    """Build once with verbose logs; cache only a checksummed, ABI-specific wheel."""\n    identity=source_wheel_identity(bundle,record,runtime,environment,profile)\n    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:20]\n    name = re.sub(r\'[-_.]+\', \'-\', record[\'package\']).lower()\n    if not re.fullmatch(r\'[a-z0-9][a-z0-9-]*\',name):raise ValueError(\'Invalid native package name\')\n    cache = Path(runtime) / \'built_source_wheels\' / name / key\n    cache.mkdir(parents=True, exist_ok=True)\n    receipt = cache / \'receipt.json\'\n    if receipt.is_file():\n        try:\n            saved = json.loads(receipt.read_text(encoding=\'utf-8\'))\n        except (ValueError, OSError):\n            saved = {}\n    else:\n        saved = {}\n    if \'wheel\' in saved:\n        wheel = checked_path(cache, saved[\'wheel\'])\n        if saved.get(\'identity\') == identity and wheel.is_file() and sha256(wheel) == saved.get(\'sha256\'):\n            print(f\'Reuse built wheel: {wheel}\', flush=True)\n            return wheel\n    packaged=Path(bundle)/\'ocr_native_cache\'/name/key\n    if (packaged/\'receipt.json\').is_file():\n        saved=json.loads((packaged/\'receipt.json\').read_text(encoding=\'utf8\'))\n        wheel=checked_path(packaged,saved[\'wheel\'])\n        if (saved.get(\'identity\')!=identity or not saved.get(\'runtime_probes_verified\') or\n                not wheel.is_file() or sha256(wheel)!=saved.get(\'sha256\')):\n            raise ValueError(f\'Invalid packaged OCR native wheel: {packaged}\')\n        with zipfile.ZipFile(wheel) as archive:\n            if archive.testzip() is not None:raise ValueError(f\'Corrupt packaged wheel: {wheel}\')\n        verify_target_wheel_tags(python,wheel)\n        destination=cache/wheel.name;shutil.copyfile(wheel,destination)\n        if sha256(destination)!=saved[\'sha256\']:raise ValueError(\'Packaged OCR native wheel copy changed\')\n        receipt.write_text(json.dumps(saved,indent=2)+\'\\n\',encoding=\'utf8\')\n        print(f\'Reuse packaged native wheel; compilation skipped: {destination}\',flush=True)\n        return destination\n    if os.environ.get(\'VTM_OCR_REQUIRE_CACHE\')==\'1\':\n        raise FileNotFoundError(f\'No compatible packaged native wheel for {name}. Run notebook 0a first; compilation disabled.\')\n    source = prepare_build_source(bundle, record, cache / \'source_runtime\')\n    if name == \'mmcv-full\':\n        print(f\'MMCV build source patches: {patch_mmcv_build_source(source)}\', flush=True)\n    logs = Path(runtime) / \'build_logs\'\n    logs.mkdir(parents=True, exist_ok=True)\n    log = logs / f\'{name}-{key}.pip.log\'\n    print(f\'Build {name}; full compiler/pip log: {log}\', flush=True)\n    with tempfile.TemporaryDirectory(prefix=\'wheel-\', dir=cache) as temporary:\n        temporary = Path(temporary).resolve()\n        if not temporary.is_relative_to(cache.resolve()):\n            raise ValueError(\'Build staging directory escapes private cache\')\n        try:\n            run([python, \'-u\', \'-m\', \'pip\', \'--log\', log, \'-vvv\', \'wheel\', \'--no-index\', \'--no-deps\',\n                 \'--no-build-isolation\', \'--no-cache-dir\', \'--no-clean\', \'--find-links\', Path(bundle) / \'wheels\',\n                 \'--wheel-dir\', temporary, source], environment)\n        except subprocess.CalledProcessError as exc:\n            if name == \'polygon3\':\n                print(\'Polygon3 direct compiler diagnostic (original pip failure is retained):\', flush=True)\n                try:\n                    run([python, \'-u\', \'setup.py\', \'build_ext\', \'--verbose\'], environment, cwd=source)\n                except subprocess.CalledProcessError:\n                    pass\n            raise RuntimeError(f\'Build {name} failed. Full compiler log: {log}; source kept at {source}\') from exc\n        wheels = list(temporary.glob(\'*.whl\'))\n        if len(wheels) != 1:\n            raise RuntimeError(f\'Build {name} produced {len(wheels)} wheels; inspect {log}\')\n        with zipfile.ZipFile(wheels[0]) as archive:\n            if archive.testzip() is not None:\n                raise ValueError(f\'Corrupt built wheel: {wheels[0]}\')\n        destination = cache / wheels[0].name\n        wheels[0].replace(destination)\n    receipt.write_text(json.dumps({\'identity\': identity, \'wheel\': destination.name,\n                                  \'sha256\': sha256(destination)}, indent=2), encoding=\'utf-8\')\n    return destination\n\n\ndef install_paddle(bundle, runtime):\n    manifest = component_manifest(bundle, "paddle")\n    directory = runtime / ("paddle_py313_cu130" if manifest.get("cuda") == "13.0" else "paddle_py313")\n    python = directory / "bin/python"\n    env = native_environment()\n    if not python.exists():\n        run(isolated_python_command(sys.executable, ["-m", "venv", "--without-pip", directory]), env)\n    if manifest.get("reused"):\n        site, version = target_site(python, runtime)\n        clear_reused_packages(site, private_kernel_cleanup_records(manifest))\n        initial_runner = private_python(python, site)\n    else:\n        initial_runner = python\n    # pip\'s --python supports a venv with no pip; the host installer routes to it.\n    run(host_pip_command(["--python", initial_runner, "install", "--no-index", "--no-deps",\n         "--find-links", bundle / "wheels", "-r", bundle / "requirements/paddle.txt"]), env)\n    reused = reuse_kernel_packages(manifest["reused"], site, version, bundle=bundle, manifest=manifest) if manifest.get("reused") else []\n    runner = private_python(python, site) if manifest.get("reused") else python\n    if manifest.get(\'reused\'):\n        ensure_private_opencv(bundle, manifest, site, runner, env, runtime)\n        run(host_pip_command([\'--python\', runner, \'check\']), env)\n    probe = """import json, paddle\nfrom paddleocr import PaddleOCR, PaddleOCRVL\nassert paddle.is_compiled_with_cuda(), \'Paddle GPU wheel required\'\nassert paddle.device.cuda.device_count() > 0, \'No CUDA device for Paddle\'\npaddle.set_device(\'gpu:0\')\nx = paddle.ones([2, 2]); y = paddle.matmul(x, x)\nprint(json.dumps({\'paddle\': paddle.__version__, \'cuda_devices\': paddle.device.cuda.device_count(), \'gpu_sum\': float(y.sum().item())}))\n"""\n    run([runner, "-c", probe], env)\n    support = install_renderer(bundle, runtime)\n    return {"paddle_python": str(runner), **support, "paddle_gpu_import_verified": True, "paddle_kernel_reused": reused}\n\n\ndef install_renderer(bundle, runtime):\n    # A common site contains only PDFium, keeping Kaggle\'s torch stack intact.\n    env = native_environment()\n    common = runtime / "common_site"\n    renderer = next((bundle / "wheels").glob("pypdfium2-5.15.0-*.whl"))\n    run(host_pip_command(["install", "--no-index", "--no-deps", "--target", common, renderer]), env)\n    return {"common_site": str(common)}\n\n\ndef install_tonggu(bundle, runtime, records):\n    manifest = component_manifest(bundle, "tonggu")\n    if manifest.get("cuda") == "13.0":\n        return install_tonggu_cu130(bundle, runtime, manifest)\n    system = json.loads((bundle / "system_assets_manifest.json").read_text(encoding="utf-8"))\n    entries = {r["package"]: r for r in system["records"]}\n    directory = runtime / "tonggu_py310"\n    python = directory / "python/bin/python3.10"\n    if not python.exists():\n        prepare_python_runtime(bundle, entries["python-runtime"], directory)\n    env = native_environment()\n    bootstrap = ["pip==25.3", "setuptools==80.10.2", "wheel==0.48.0", "Cython==3.0.12", "ninja==1.13.2", "numpy==2.2.6"]\n    run([python, "-m", "pip", "install", "--no-index", "--no-deps", "--find-links", bundle / "wheels", *bootstrap], env)\n    run([python, "-m", "pip", "install", "--no-index", "--no-deps", "--no-build-isolation",\n         "--find-links", bundle / "wheels", "--find-links", bundle / "sources",\n         "-r", bundle / "requirements/tonggu.txt"], env)\n\n    cuda = runtime / "cuda12.8"\n    for name in ("cuda_nvcc", "cuda_cudart", "cuda_cccl"):\n        extracted = runtime / ("extract_" + name)\n        extract_tar(checked_path(bundle, entries[name]["path"]), extracted)\n        roots = [p for p in extracted.iterdir() if p.is_dir()]\n        if len(roots) != 1:\n            raise ValueError(f"Unexpected CUDA archive layout: {name}")\n        shutil.copytree(roots[0], cuda, dirs_exist_ok=True)\n    env = native_environment(cuda)\n    run([cuda / "bin/nvcc", "--version"], env)\n    # Some CUDA redistributions call the runtime directory \'lib\', others \'lib64\'.\n    if not (cuda / "lib64").exists() and (cuda / "lib").exists():\n        (cuda / "lib64").symlink_to("lib", target_is_directory=True)\n\n    repo = prepare_tonggu_checkout(bundle, entries[\'tonggu-source\'], runtime)\n    built = runtime / "built_wheels"\n    built.mkdir(exist_ok=True)\n    mmcv_source = checked_path(bundle, entries["mmcv-full"]["path"])\n    if not list(built.glob("mmcv_full-1.7.2-*.whl")):\n        run([python, "-m", "pip", "wheel", "--no-index", "--no-deps", "--no-build-isolation",\n             "--find-links", bundle / "wheels", "--wheel-dir", built, mmcv_source],\n            {**env, "MMCV_WITH_OPS": "1", "PIP_NO_BUILD_ISOLATION": "1"})\n    mmcv = next(built.glob("mmcv_full-1.7.2-*.whl"))\n    run([python, "-m", "pip", "install", "--no-index", "--no-deps", mmcv], env)\n    run([python, repo / "tools/patch_openmmlab.py"], env)\n    probe = """import torch, mmcv, mmcv.ops, mmdet, mmrotate, json\nassert torch.version.cuda.startswith(\'12.8\'), torch.version.cuda\nboxes = torch.tensor([[5., 5., 2., 2., 0.]], device=\'cuda\')\nscores = torch.tensor([.9], device=\'cuda\')\n_, keep = mmcv.ops.nms_rotated(boxes, scores, .5)\nassert keep.numel() == 1\nprint(json.dumps({\'torch\': torch.__version__, \'cuda\': torch.version.cuda, \'mmcv\': mmcv.__version__, \'sm\': list(torch.cuda.get_device_capability()), \'rotated_nms_gpu\': \'OK\'}))\n"""\n    run([python, "-c", probe], env)\n    return {"tonggu_python": str(python), "tonggu_repo": str(repo),\n            "tonggu_compiled_ops_verified": True, "tonggu_cuda_home": str(cuda)}\n\n\ndef install_tonggu_cu130(bundle, runtime, manifest):\n    system = component_manifest(bundle, "system_assets")\n    entries = {record["package"]: record for record in system["records"]}\n    directory = runtime / "tonggu_py310_cu130"\n    python = directory / "python/bin/python3.10"\n    if not python.exists():\n        prepare_python_runtime(bundle, entries["python-runtime"], directory)\n    nvcc = shutil.which("nvcc")\n    if not nvcc:\n        raise RuntimeError("TongGu needs the CUDA 13.0 toolkit verified in the Kaggle report")\n    candidates = [Path(value) for value in (os.environ.get("CUDA_HOME"), os.environ.get("CUDA_PATH")) if value]\n    candidates += [Path(nvcc).resolve().parents[1], Path("/usr/local/cuda"), Path("/usr/local/cuda-13.0")]\n    cuda = next((path.resolve() for path in candidates if (path / "include/cuda.h").is_file()), candidates[0].resolve())\n    compiler = subprocess.check_output([nvcc, "--version"], text=True)\n    if "release 13.0" not in compiler or not (cuda / "include/cuda.h").is_file():\n        raise RuntimeError(f"Expected a complete CUDA 13.0 toolkit, found {cuda}")\n    env = native_environment(cuda)\n    site, version = target_site(python, runtime)\n    env, toolchain = prepare_build_toolchain(python, runtime, env)\n    # A package promoted from kernel reuse must lose its previous private\n    # metadata too, otherwise pip may incorrectly treat a partial copy as installed.\n    clear_reused_packages(site, private_kernel_cleanup_records(manifest))\n    initial_runner = private_python(python, site)\n    run(host_pip_command(["--python", initial_runner, "install", "--no-index", "--no-deps",\n         "--find-links", bundle / "wheels", "-r", bundle / "requirements/tonggu.txt"]), env)\n    reused = reuse_kernel_packages(manifest.get("reused", []), site, version, bundle=bundle, manifest=manifest)\n    runner = private_python(python, site)\n    ensure_private_opencv(bundle, manifest, site, runner, env, runtime)\n    # Check the selected Torch/CUDA before spending time compiling mmcv.\n    torch_probe = "import torch; assert torch.version.cuda == \'13.0\', torch.version.cuda; x=torch.ones((2,2),device=\'cuda\'); assert float((x@x).sum().item()) == 8.0; print(torch.__version__,torch.cuda.get_device_capability())"\n    run([runner, "-c", torch_probe], env)\n    configure_cuda_architecture(runner, env)\n    versions = {record[\'package\']: record[\'version\'] for record in [*manifest[\'records\'], *reused]}\n    profile = {\'python\': version, \'numpy\': versions.get(\'numpy\'), \'torch\': versions.get(\'torch\'),\n               \'cuda\': \'13.0\', \'arch\': env[\'TORCH_CUDA_ARCH_LIST\'],\n               \'toolchain\': {k: v for k, v in toolchain.items() if k != \'changed_keys\'}}\n    built_wheels=[]\n    for record in manifest["records"]:\n        if record["kind"] != "wheel" and record["package"] != "mmcv-full":\n            wheel = build_offline_source_wheel(bundle, record, runtime, runner, env, profile)\n            built_wheels.append(str(wheel))\n            run([runner, \'-m\', \'pip\', \'install\', \'--no-index\', \'--no-deps\', \'--force-reinstall\', wheel], env)\n    source_probe = """import Polygon, numpy as np, jieba, zhconv, cv2\nassert Polygon.withNumPy, \'Polygon must be built with the selected NumPy headers\'\np = Polygon.Polygon(np.array([[0.,0.],[2.,0.],[2.,2.],[0.,2.]], dtype=np.float64))\nassert abs(p.area()-4.) < 1e-9\nassert cv2.resize(np.zeros((2,2,3), dtype=np.uint8),(4,4)).shape == (4,4,3)\nprint(\'Polygon/NumPy, jieba, zhconv, OpenCV native imports/operations OK\')\n"""\n    run([runner, \'-c\', source_probe], env)\n    repo = prepare_tonggu_checkout(bundle, entries[\'tonggu-source\'], runtime)\n    mmcv_source = next(record for record in manifest["records"] if record["package"] == "mmcv-full")\n    mmcv = build_offline_source_wheel(bundle, mmcv_source, runtime, runner,\n        {**env, "MMCV_WITH_OPS": "1", "PIP_NO_BUILD_ISOLATION": "1"}, profile)\n    built_wheels.append(str(mmcv))\n    run([runner, "-m", "pip", "install", "--no-index", "--no-deps", \'--force-reinstall\', mmcv], env)\n    run([runner, repo / "tools/patch_openmmlab.py"], env)\n    run([runner, \'-m\', \'pip\', \'check\'], env)\n    probe = """import torch, mmcv, mmcv.ops, mmdet, mmrotate, json\nb = torch.tensor([[5.,5.,2.,2.,0.]], device=\'cuda\')\ns = torch.tensor([.9], device=\'cuda\')\n_, keep = mmcv.ops.nms_rotated(b,s,.5)\nassert keep.numel() == 1\nprint(json.dumps({\'torch\':torch.__version__,\'cuda\':torch.version.cuda,\'mmcv\':mmcv.__version__,\'rotated_nms_gpu\':\'OK\'}))\n"""\n    run([runner, "-c", probe], env)\n    return {"tonggu_python": str(runner), "tonggu_repo": str(repo), "tonggu_cuda_home": str(cuda),\n            "tonggu_compiled_ops_verified": True, "tonggu_kernel_reused": reused,\n            "tonggu_built_wheels":built_wheels}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--bundle", type=Path, required=True)\n    parser.add_argument("--runtime", type=Path, default=Path("/tmp/vtm_kaggle_runtime"))\n    parser.add_argument("--component", choices=["app", "paddle", "tonggu", "all"], default="paddle")\n    parser.add_argument("--verify-only", action="store_true")\n    args = parser.parse_args()\n    if sys.platform != "linux" or platform.machine() != "x86_64" or sys.version_info[:2] != (3, 13):\n        raise SystemExit("Run this installer with Kaggle Linux x86_64 Python 3.13; it does not install on Windows")\n    bundle, runtime = args.bundle.resolve(), args.runtime.resolve()\n    records = verify(bundle)\n    if args.verify_only:\n        return\n    runtime.mkdir(parents=True, exist_ok=True)\n    configuration = runtime / "runtime_config.json"\n    data = json.loads(configuration.read_text(encoding="utf-8")) if configuration.exists() else {}\n    data.update(bundle_root=str(bundle))\n    if args.component == "app":\n        data.update(install_renderer(bundle, runtime))\n        configuration.write_text(json.dumps(data, indent=2), encoding="utf-8")\n    if args.component in {"paddle", "all"}:\n        data.update(install_paddle(bundle, runtime))\n        configuration.write_text(json.dumps(data, indent=2), encoding="utf-8")\n    if args.component in {"tonggu", "all"}:\n        data.update(install_tonggu(bundle, runtime, records))\n        configuration.write_text(json.dumps(data, indent=2), encoding="utf-8")\n    print("Runtime configuration:", configuration)\n    print(json.dumps(data, indent=2))\n    print("Packages/GPU ops only: OCR weights and PDF data still need separate offline Datasets.")\n\n\nif __name__ == "__main__":\n    main()\n'
OCR_PACKAGE_SOURCE = '"""Build/export TongGu native wheels on Kaggle; no OCR models or PDFs."""\nfrom __future__ import annotations\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport signal\nimport subprocess\nimport sys\nimport time\nimport zipfile\n\n\ndef file_hash(path):\n    digest=hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda:stream.read(8*1024**2),b\'\'):digest.update(block)\n    return digest.hexdigest()\n\n\ndef export_ocr_native_package(packages_dir,runtime_dir,output_dir):\n    runtime=Path(runtime_dir).resolve();output=Path(output_dir);output.mkdir(parents=True,exist_ok=True)\n    state=json.loads((runtime/\'runtime_config.json\').read_text(encoding=\'utf8\'))\n    if not state.get(\'tonggu_compiled_ops_verified\'):\n        raise ValueError(\'TongGu CUDA/source probes have not passed; no verified package will be exported\')\n    wheels=state.get(\'tonggu_built_wheels\',[])\n    if not wheels:raise ValueError(\'No native wheel receipts from the current installer\')\n    manifest=json.loads((Path(packages_dir)/\'tonggu_manifest.json\').read_text(encoding=\'utf8\'))\n    sources={r[\'package\']:r for r in manifest[\'records\'] if r[\'kind\']!=\'wheel\'}\n    exported=[];payload=[]\n    for value in wheels:\n        wheel=Path(value).resolve()\n        if not wheel.is_relative_to(runtime/\'built_source_wheels\'):\n            raise ValueError(\'Native wheel is outside the private build cache\')\n        saved=json.loads((wheel.parent/\'receipt.json\').read_text(encoding=\'utf8\'))\n        identity=saved[\'identity\'];source=sources.get(identity[\'package\'])\n        if (not source or source[\'sha256\']!=identity[\'source_sha256\'] or\n                source[\'version\']!=identity[\'version\'] or saved[\'wheel\']!=wheel.name or\n                file_hash(wheel)!=saved[\'sha256\']):raise ValueError(\'Native wheel receipt/source identity mismatch\')\n        if identity.get(\'build_schema\')!=3:raise ValueError(\'Re-run the updated installer to get portable wheel identities\')\n        with zipfile.ZipFile(wheel) as archive:\n            if archive.testzip() is not None:raise ValueError(\'Native wheel ZIP integrity failed\')\n        name=re.sub(r\'[-_.]+\',\'-\',identity[\'package\']).lower()\n        if not re.fullmatch(r\'[a-z0-9][a-z0-9-]*\',name):raise ValueError(\'Invalid native package name\')\n        key=hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()[:20]\n        prefix=f\'ocr_native_cache/{name}/{key}/\'\n        saved=dict(saved,runtime_probes_verified=True)\n        payload.append((wheel,prefix+wheel.name,json.dumps(saved,indent=2)+\'\\n\',prefix+\'receipt.json\'))\n        exported.append(dict(package=identity[\'package\'],wheel=wheel.name,sha256=saved[\'sha256\'],profile=identity[\'profile\']))\n    if not any(r[\'package\']==\'mmcv-full\' for r in exported):raise ValueError(\'Verified MMCV wheel missing\')\n    report=dict(format=\'vtm-ocr-native-package-v1\',status=\'native_and_cuda_verified\',\n        tonggu_compiled_ops_verified=True,model_weights_loaded=False,ocr_inference_verified=False,\n        kernel_modified=False,wheels=exported)\n    report_text=json.dumps(report,indent=2)+\'\\n\'\n    (output/\'ocr_native_build_report.json\').write_text(report_text,encoding=\'utf8\')\n    target=output/\'ocr_native_package.zip\';temporary=target.with_suffix(\'.zip.tmp\')\n    with zipfile.ZipFile(temporary,\'w\',compression=zipfile.ZIP_STORED) as archive:\n        for wheel,name,receipt,receipt_name in payload:\n            archive.write(wheel,name);archive.writestr(receipt_name,receipt)\n        archive.writestr(\'ocr_native_cache/build_report.json\',report_text)\n    with zipfile.ZipFile(temporary) as archive:\n        if archive.testzip() is not None:raise ValueError(\'OCR package ZIP integrity failed\')\n    temporary.replace(target)\n    return dict(report,archive=str(target),archive_sha256=file_hash(target),install_under=\'kaggle_upload/packages\')\n\n\ndef build_ocr_native_package(packages_dir,runtime_dir,output_dir,installer_path,*,allow_build=True,total_timeout=21600):\n    if sys.platform!=\'linux\':raise RuntimeError(\'Run native package preparation on Kaggle Linux\')\n    output=Path(output_dir);output.mkdir(parents=True,exist_ok=True)\n    log=output/\'ocr_native_build.log\';started=time.monotonic()\n    env=dict(os.environ,PIP_NO_INDEX=\'1\',PIP_DISABLE_PIP_VERSION_CHECK=\'1\',PYTHONUNBUFFERED=\'1\',\n             VTM_OCR_REQUIRE_CACHE=\'0\' if allow_build else \'1\')\n    command=[sys.executable,\'-u\',str(installer_path),\'--bundle\',str(packages_dir),\n             \'--runtime\',str(runtime_dir),\'--component\',\'tonggu\']\n    print(f\'TongGu native setup; full log: {log}\',flush=True)\n    process=None\n    try:\n        with log.open(\'w\',encoding=\'utf8\') as stream:\n            process=subprocess.Popen(command,env=env,stdout=stream,stderr=subprocess.STDOUT,start_new_session=True)\n            notice=started\n            while process.poll() is None:\n                elapsed=time.monotonic()-started\n                if elapsed>total_timeout:raise TimeoutError(f\'OCR native setup exceeded {total_timeout}s; see {log}\')\n                if time.monotonic()-notice>=30:\n                    with log.open(\'rb\') as reader:\n                        reader.seek(0,2);reader.seek(max(0,reader.tell()-4096))\n                        tail=reader.read().decode(\'utf8\',errors=\'replace\').splitlines()\n                    print(f\'OCR native setup: {elapsed:.0f}s; \'+(tail[-1][-240:] if tail else \'starting\'),flush=True)\n                    notice=time.monotonic()\n                time.sleep(.5)\n        if process.returncode:raise RuntimeError(f\'TongGu setup failed (exit {process.returncode}); see {log}\')\n        result=export_ocr_native_package(packages_dir,runtime_dir,output)\n        result[\'seconds\']=round(time.monotonic()-started,3)\n        return result\n    except BaseException as exc:\n        if process is not None and process.poll() is None:\n            try:os.killpg(process.pid,signal.SIGTERM)\n            except ProcessLookupError:pass\n            try:process.wait(timeout=10)\n            except subprocess.TimeoutExpired:\n                try:os.killpg(process.pid,signal.SIGKILL)\n                except ProcessLookupError:pass\n                process.wait()\n        failure=dict(status=\'failed\',error=str(exc),log=str(log),model_weights_loaded=False,ocr_inference_verified=False)\n        (output/\'ocr_native_build_report.json\').write_text(json.dumps(failure,indent=2)+\'\\n\',encoding=\'utf8\')\n        raise\n'
OCR_INSTALLER_SHA256 = 'a1a695ced9a7117724a0c13d8df26fc4e72fe12444bc6caeb63f783a407330b0'
OCR_PACKAGE_SHA256 = 'f1134773db14d079c8c2b6903f30afb73a01d280970880300dcb6945728290ee'
for data,expected in ((OCR_INSTALLER_SOURCE,OCR_INSTALLER_SHA256),(OCR_PACKAGE_SOURCE,OCR_PACKAGE_SHA256)):
    if hashlib.sha256(data.encode('utf8')).hexdigest()!=expected:raise ValueError('OCR helper checksum mismatch')
ocr_bootstrap=Path(OCR_RUNTIME_DIR)/'bootstrap';ocr_bootstrap.mkdir(parents=True,exist_ok=True)
OCR_INSTALLER_PATH=ocr_bootstrap/('install_ocr_'+OCR_INSTALLER_SHA256[:16]+'.py')
OCR_INSTALLER_PATH.write_text(OCR_INSTALLER_SOURCE,encoding='utf8')
ocr_helper=ocr_bootstrap/('package_ocr_'+OCR_PACKAGE_SHA256[:16]+'.py')
ocr_helper.write_text(OCR_PACKAGE_SOURCE,encoding='utf8')
ocr_spec=importlib.util.spec_from_file_location('vtm_ocr_native_package',ocr_helper)
OCR_PACKAGE=importlib.util.module_from_spec(ocr_spec);ocr_spec.loader.exec_module(OCR_PACKAGE)


In [ ]:
if not (PREPARE_FLASHATTENTION or PREPARE_NOTEBOOK3):raise ValueError('Bật ít nhất một phần chuẩn bị')
if not PACKAGES_DIR:
    groups=[]
    if PREPARE_FLASHATTENTION:
        groups.append({str(p.parent.parent.parent) for p in Path('/kaggle/input').rglob('unihir_runtime.json')
            if p.parent.name=='manifests' and p.parent.parent.name=='image_restoration'})
    if PREPARE_NOTEBOOK3:groups.append({str(p.parent) for p in Path('/kaggle/input').rglob('tonggu_manifest.json')})
    candidates=sorted(set.intersection(*groups))
    if len(candidates)!=1:raise RuntimeError('Điền PACKAGES_DIR; các gốc phù hợp: '+repr(candidates))
    PACKAGES_DIR=candidates[0]
if not Path(PACKAGES_DIR).is_dir():raise FileNotFoundError(PACKAGES_DIR)
os.environ['VTM_FLASH_MAX_JOBS']=str(MAX_BUILD_JOBS)
os.environ['VTM_FLASH_NVCC_THREADS']=str(NVCC_THREADS)
resources=FLASH_RUNTIME.native_build_resources()
os.environ['MAX_JOBS']=str(resources['max_jobs'])
print('Packages:',PACKAGES_DIR,'; compiler resources:',resources,flush=True)
if CACHE_INPUT:os.environ['VTM_FLASH_CACHE_INPUT']=CACHE_INPUT
else:os.environ.pop('VTM_FLASH_CACHE_INPUT',None)
RESULTS={};ERRORS={}
if PREPARE_FLASHATTENTION:
    try:
        RESULTS['flashattention']=FLASH_RUNTIME.build_flash_package(PACKAGES_DIR,RUNTIME_DIR,OUTPUT_DIR,allow_build=ALLOW_BUILD)
        print(json.dumps(RESULTS['flashattention'],ensure_ascii=False,indent=2))
    except Exception as exc:
        ERRORS['flashattention']=str(exc);print('FlashAttention lỗi:',exc,flush=True)
else:print('FlashAttention: bỏ qua theo cấu hình')

In [ ]:
if PREPARE_NOTEBOOK3:
    try:
        RESULTS['notebook3']=OCR_PACKAGE.build_ocr_native_package(
            PACKAGES_DIR,OCR_RUNTIME_DIR,OCR_OUTPUT_DIR,OCR_INSTALLER_PATH,
            allow_build=ALLOW_BUILD,total_timeout=OCR_BUILD_TIMEOUT_SECONDS)
        print(json.dumps(RESULTS['notebook3'],ensure_ascii=False,indent=2))
    except Exception as exc:
        ERRORS['notebook3']=str(exc);print('Notebook 3 native setup lỗi:',exc,flush=True)
else:print('Notebook 3 native setup: bỏ qua theo cấu hình')

## Tải về và đặt đúng chỗ

Tải **`flash_attention_package.zip`** cho notebook 1 và **`ocr_native_package.zip`** cho notebook 3.
Phần tắt hoặc thất bại không có gói thành công mới. Hai phần chạy độc lập và báo lỗi ở ô cuối.
Giữ nguyên tên ZIP; giải nén **cả hai ZIP trực tiếp vào cùng thư mục**:

```text
C:\Users\mduy\source\chinese-vietnamese-vtm-corpus\kaggle_upload\packages\
```

Cấu trúc sau giải nén:

```text
packages/
    image_restoration/flash_attention_cache/<hash>/
        flash_attn-....whl
        flash_wheel.json
        build_report.json
    ocr_native_cache/<package>/<hash>/
        <wheel>.whl
        receipt.json
    ocr_native_cache/build_report.json
```

Không đổi tên wheel, package, thư mục hash hoặc lồng thêm folder tên ZIP.
Cập nhật Dataset packages trên Kaggle, gắn phiên bản mới vào notebook 1/3 đã cập nhật.
Cả hai notebook tự tìm cache trong packages; không cần chuyển cache sang models.
Notebook 1/3 mặc định chỉ dùng cache đã có, dừng sớm khi không tương thích thay vì tự build lâu.
FlashAttention cp313 không dùng thay MMCV cp310. Giữ source/manifest gốc để kiểm tra hash.
Report chỉ chứng minh setup và toán tử native/GPU; chưa chứng minh OCR hay UniHIR inference.

In [ ]:
from IPython.display import FileLink, display
for label,result in RESULTS.items():
    path=Path(result['archive'])
    print(label,path)
    display(FileLink(str(path)))
if ERRORS:
    print(json.dumps(ERRORS,ensure_ascii=False,indent=2))
    raise RuntimeError('Một số phần chuẩn bị thất bại; giữ gói đã thành công và xem report/log trong Output')